In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import re

# Load your data
df = pd.read_csv('C:/Users/DELL/Downloads/Legal Case/processed_data/clean_legal_cases.csv')
df.head()

,Case_ID,Case_Name,Source_Folder,Year,Court,Case_Text,Case_Text_Full_Length,Verdict,Case_Type,Sub_Type,Num_Citations,Legal_Citations,Decision_Date,Docket_Number,First_Page,Last_Page
0,347,0347-01,fil3212,1959,Connecticut Superior Court,James Yaworski et al. v. Town of Canterbury et...,8903,Granted,Criminal Law,General Criminal,12,137 Conn. 701; 14 Conn. Sup. 202; 143 Conn. 15...,1959-05-21,File No. 11074,347,352
1,477,0477-01,fil3212,1958,Connecticut Superior Court,State of Connecticut v. John Salta Decided Oct...,2851,Judgment AFFIRMED,Criminal Law,Assault & Battery,1,21 Conn. Supp. 477,1958-10-17,NaN,477,479
2,217,0217-01,fil3212,1959,Connecticut Superior Court,"The Bacon Memorial Home v. John J. Bracken, At...",8245,Judgment Entered (No Costs),Civil Procedure,Appeal,11,100 Conn. 5; 116 Conn. 347; 120 Conn. 77; 139 ...,1959-03-02,File No. 25602,217,222
3,132,0132-01,fil3212,1958,Connecticut Superior Court,Hazel M. Harris v. Housing Authority of the Ci...,2903,Demurrer SUSTAINED,Civil Procedure,Demurrer,7,117 Conn. 398; 120 Conn. 577; 21 Conn. Sup. 65...,1958-10-21,File No. 23885,132,134
4,487,0487-01,fil3212,1960,Connecticut Superior Court,"Yolande Berube v. The Salvation Army, Inc. Sup...",1483,Sustained,Civil Procedure,Demurrer,2,113 Conn. 188; 21 Conn. Supp. 487,1960-01-05,File No. 117954,487,488


In [2]:
import pandas as pd

# Count Sub_Type occurrences
subtype_counts = df["Case_Type"].value_counts()

# Percentage distribution
subtype_percentage = df["Case_Type"].value_counts(normalize=True) * 100

# Combine into one table
subtype_distribution = pd.DataFrame({
    "Count": subtype_counts,
    "Percentage (%)": subtype_percentage.round(2)
})

print(subtype_distribution)

                               Count  Percentage (%)
Case_Type                                           
Civil Procedure                 5364           33.34
Criminal Law                    4153           25.81
Contract Law - Debt             2412           14.99
Property Law - Ejectment        1895           11.78
Torts                           1841           11.44
Torts - Defamation               221            1.37
Property Law - Execution Sale    205            1.27


In [3]:
import pandas as pd

# Count Sub_Type occurrences
subtype_counts = df["Sub_Type"].value_counts()

# Percentage distribution
subtype_percentage = df["Sub_Type"].value_counts(normalize=True) * 100

# Combine into one table
subtype_distribution = pd.DataFrame({
    "Count": subtype_counts,
    "Percentage (%)": subtype_percentage.round(2)
})

print(subtype_distribution)

                         Count  Percentage (%)
Sub_Type                                      
Appeal                    3614           22.46
General Criminal          3273           20.34
Demurrer                  1271            7.90
Negligence                1269            7.89
Title Dispute             1057            6.57
Promissory Note            868            5.39
Bond                       743            4.62
General Torts              435            2.70
Debt Collection            382            2.37
Mortgage Foreclosure       358            2.22
Homicide                   310            1.93
Pleading                   295            1.83
General Property           262            1.63
General Contract           253            1.57
Larceny                    237            1.47
Ejectment                  203            1.26
Usury                      156            0.97
General Execution          144            0.89
Slander                    140            0.87
Assault & Bat

In [4]:
def merge_subtypes(st):
    st = str(st).lower()

    # APPEAL
    if "appeal" in st:
        return "APPELLATE_PROCEDURE"

    # CIVIL PROCEDURE
    if st in ["demurrer", "pleading", "writ", "capias", "scire facias", "general civil procedure"]:
        return "CIVIL_PROCEDURE"

    # CONTRACT / DEBT
    if st in ["promissory note", "bond", "debt collection", "usury", "general contract"]:
        return "CONTRACT_DEBT"

    # PROPERTY
    if st in ["ejectment", "title dispute", "general property", "mortgage foreclosure"]:
        return "PROPERTY_DISPUTE"

    # TORT
    if st in ["slander", "libel", "defamation", "negligence", "trespass", "conversion", "general torts"]:
        return "TORT"

    # CRIMINAL
    if st in ["general criminal", "larceny", "homicide", "assault & battery",
              "indictment", "perjury", "riot", "counterfeiting",
              "burglary/robbery", "affray"]:
        return "CRIMINAL_OFFENSE"

    # EXECUTION
    if st in ["fieri facias", "general execution", "attachment"]:
        return "EXECUTION_ENFORCEMENT"

    return "OTHER"

In [5]:
df["SubType_Merged_Label"] = df["Sub_Type"].apply(merge_subtypes)
print(df["SubType_Merged_Label"].value_counts())

SubType_Merged_Label
CRIMINAL_OFFENSE         4160
APPELLATE_PROCEDURE      3614
CONTRACT_DEBT            2402
TORT                     2055
PROPERTY_DISPUTE         1880
CIVIL_PROCEDURE          1729
EXECUTION_ENFORCEMENT     199
OTHER                      52
Name: count, dtype: int64


In [6]:
"""
Sub-Type Classification Pipeline
==================================
Collapses 36 granular Sub_Type values into 8 broad ML-ready groups.

The 8 broad groups:
    PROCEDURAL       — Appeal, Demurrer, and all procedural writs
    GENERAL_CRIMINAL — General Criminal, Indictment
    VIOLENT_CRIME    — Homicide, Assault & Battery, Riot, Affray, Burglary/Robbery
    PROPERTY_CRIME   — Larceny, Counterfeiting, Perjury
    DEBT_CONTRACT    — Promissory Note, Bond, Debt Collection, Usury, General Contract
    PROPERTY_CIVIL   — Ejectment, Title Dispute, Mortgage Foreclosure, Fieri Facias, etc.
    DEFAMATION_TORT  — Slander, Defamation, Libel, Trespass, Negligence, etc.
    GENERAL_CIVIL    — General Civil Procedure
    OTHER            — safe fallback for any unmapped value — never NaN

Recommended ML model for this task:
    Use a multi-class classifier such as:
      - Logistic Regression (fast baseline, works well with TF-IDF on Case_Text)
      - Random Forest / XGBoost (handles categorical + numeric features well)
      - BERT fine-tuned classifier (best accuracy if Case_Text is the main signal)
    Target column  : Sub_Type_Group (8-class)
    Key features   : Case_Text (NLP), Case_Type_Group (categorical), Court, Year
"""

import re
import pandas as pd
import numpy as np
from typing import Tuple, Dict, List


# ═══════════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ═══════════════════════════════════════════════════════════════════════════════

# All 8 broad sub-type group labels (ordered by size for reference)
SUB_TYPE_GROUPS = [
    'PROCEDURAL',       # ~50%  — Appeal + Demurrer + procedural writs
    'GENERAL_CRIMINAL', # ~21%  — General Criminal + Indictment
    'DEBT_CONTRACT',    # ~10%  — Promissory Note + Bond + Debt Collection etc.
    'PROPERTY_CIVIL',   # ~7%   — Ejectment + Title Dispute + Mortgage etc.
    'DEFAMATION_TORT',  # ~4%   — Slander + Defamation + Libel + Negligence etc.
    'VIOLENT_CRIME',    # ~5%   — Homicide + Assault + Riot + Burglary etc.
    'PROPERTY_CRIME',   # ~2.5% — Larceny + Counterfeiting + Perjury
    'GENERAL_CIVIL',    # ~0.2% — General Civil Procedure
    'OTHER',            # safe fallback — never NaN
]

# Columns that are leakage sources — must NOT appear in X at train time
LEAKAGE_COLUMNS = [
    'verdict_text',
    'verdict_clean',
    'keyword',
    'verdict_category',
    'outcome_simple',
    'four_class_label',
    'Sub_Type',          # raw sub-type — replaced by Sub_Type_Group
    'Sub_Type_Group',    # the new target column
]

# The 5 ML input features — used verbatim as X
REQUIRED_FEATURES = ['Case_Text', 'Case_Type', 'Sub_Type_Group', 'Court', 'Year']


# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1 — SUB-TYPE MERGING MAP
# Maps each of the 36 raw Sub_Type strings → one of 8 broad groups.
# Add new Sub_Type values here as they appear in your data.
# Unmapped values fall to 'OTHER' — never dropped, never NaN.
# ═══════════════════════════════════════════════════════════════════════════════

# ── RECOMMENDED ML CLASS ──────────────────────────────────────────────────────
# For this 8-class sub-type classification task, the recommended approach is:
#   1. TF-IDF on Case_Text  +  one-hot encoded Case_Type / Court  +  Year
#      → feed into Logistic Regression or XGBoost (strong baseline)
#   2. Fine-tuned BERT / Legal-BERT on Case_Text
#      → best accuracy for text-heavy signal
#   Class imbalance: PROCEDURAL dominates at ~50%.
#   Use class_weight='balanced' or oversample minority classes.
# ─────────────────────────────────────────────────────────────────────────────

SUB_TYPE_MAPPING: Dict[str, str] = {

    # ── PROCEDURAL (~50%) ─────────────────────────────────────────────────────
    # Procedural posture — cuts across all case types
    # Appeal and Demurrer alone cover ~48% of all rows
    'Appeal':             'PROCEDURAL',
    'Demurrer':           'PROCEDURAL',
    'Pleading':           'PROCEDURAL',  # rare (0.15%) — procedural pleading disputes
    'Writ':               'PROCEDURAL',  # rare (0.05%) — writ proceedings
    'Capias':             'PROCEDURAL',  # rare (0.07%) — writ of capias (arrest order)
    'Scire Facias':       'PROCEDURAL',  # rare (0.02%) — procedural writ to enforce judgment
    'Attachment':         'PROCEDURAL',  # rare (0.02%) — pre-judgment asset seizure order

    # ── GENERAL_CRIMINAL (~21%) ───────────────────────────────────────────────
    # Unspecified criminal matters and formal charging documents
    'General Criminal':   'GENERAL_CRIMINAL',
    'Indictment':         'GENERAL_CRIMINAL',  # formal criminal charge — same domain

    # ── VIOLENT_CRIME (~5.4%) ─────────────────────────────────────────────────
    # Physical violence or forcible crimes against persons
    'Homicide':           'VIOLENT_CRIME',
    'Assault & Battery':  'VIOLENT_CRIME',
    'Riot':               'VIOLENT_CRIME',
    'Affray':             'VIOLENT_CRIME',       # rare (0.02%) — public fighting
    'Burglary/Robbery':   'VIOLENT_CRIME',

    # ── PROPERTY_CRIME (~2.5%) ────────────────────────────────────────────────
    # Non-violent crimes involving theft, fraud, or false statements
    'Larceny':            'PROPERTY_CRIME',
    'Counterfeiting':     'PROPERTY_CRIME',
    'Perjury':            'PROPERTY_CRIME',      # false statement — same fraud domain

    # ── DEBT_CONTRACT (~10.3%) ────────────────────────────────────────────────
    # Obligations, instruments, and enforcement of contracts
    'Promissory Note':    'DEBT_CONTRACT',
    'Bond':               'DEBT_CONTRACT',
    'Debt Collection':    'DEBT_CONTRACT',
    'Usury':              'DEBT_CONTRACT',        # illegal interest on debt — same domain
    'General Contract':   'DEBT_CONTRACT',

    # ── PROPERTY_CIVIL (~7.1%) ────────────────────────────────────────────────
    # Civil disputes over real or personal property rights
    'Ejectment':          'PROPERTY_CIVIL',
    'Title Dispute':      'PROPERTY_CIVIL',
    'Mortgage Foreclosure': 'PROPERTY_CIVIL',
    'General Property':   'PROPERTY_CIVIL',
    'Fieri Facias':       'PROPERTY_CIVIL',      # writ to seize property for debt
    'General Execution':  'PROPERTY_CIVIL',      # enforcement of property judgment

    # ── DEFAMATION_TORT (~4.2%) ───────────────────────────────────────────────
    # Civil wrongs — harm to reputation, person, or property
    'Slander':            'DEFAMATION_TORT',
    'Defamation':         'DEFAMATION_TORT',
    'Libel':              'DEFAMATION_TORT',
    'Trespass':           'DEFAMATION_TORT',     # civil trespass — tort domain
    'Negligence':         'DEFAMATION_TORT',
    'Conversion':         'DEFAMATION_TORT',     # civil taking of property
    'General Torts':      'DEFAMATION_TORT',

    # ── GENERAL_CIVIL (~0.2%) ─────────────────────────────────────────────────
    # Catch-all civil procedure with no further classification
    'General Civil Procedure': 'GENERAL_CIVIL',
}


# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2 — APPLY SUB-TYPE MAPPING
# ═══════════════════════════════════════════════════════════════════════════════

def apply_sub_type_mapping(
    df: pd.DataFrame,
    sub_type_col: str = 'Sub_Type',
) -> pd.DataFrame:
    """
    Map each raw Sub_Type value to a broad group using SUB_TYPE_MAPPING.

    Adds a new column 'Sub_Type_Group' — this becomes an ML input feature.
    The original Sub_Type column is kept for audit but must not enter X.
    Unmapped values are set to 'OTHER' — never NaN.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain a column named `sub_type_col`.
    sub_type_col : str
        Name of the raw sub-type column. Default 'Sub_Type'.

    Returns
    -------
    pd.DataFrame
        df with new column 'Sub_Type_Group' appended.

    Raises
    ------
    KeyError
        If `sub_type_col` is not in df.
    """
    if sub_type_col not in df.columns:
        raise KeyError(
            f"Column '{sub_type_col}' not found. "
            f"Available columns: {list(df.columns)}"
        )

    df = df.copy()

    # Map to broad group; anything not in the dict becomes 'OTHER'
    df['Sub_Type_Group'] = df[sub_type_col].map(SUB_TYPE_MAPPING).fillna('OTHER')

    # ── Integrity check — zero NaN tolerance ─────────────────────────────────
    nan_count = df['Sub_Type_Group'].isna().sum()
    if nan_count > 0:
        raise ValueError(f"Sub_Type_Group has {nan_count} NaN values after mapping.")

    # ── Distribution report ───────────────────────────────────────────────────
    dist  = df['Sub_Type_Group'].value_counts()
    total = len(df)

    print(f"\n{'='*55}")
    print(f"  SUB-TYPE GROUP DISTRIBUTION AFTER MERGING  (n={total:,})")
    print(f"{'='*55}")
    for grp in SUB_TYPE_GROUPS:
        n   = dist.get(grp, 0)
        pct = n / total * 100
        bar = '█' * int(pct / 2)
        print(f"  {grp:<18}  {n:>5}  {pct:5.1f}%  {bar}")
    print(f"  {'─'*50}")
    print(f"  {'TOTAL':<18}  {total:>5}  100.0%")

    # Warn if any values fell through to OTHER
    other_n = dist.get('OTHER', 0)
    if other_n > 0:
        other_vals = df.loc[df['Sub_Type_Group'] == 'OTHER', sub_type_col].value_counts()
        print(f"\n  [WARN] {other_n} rows mapped to OTHER.")
        print(f"  Unmapped Sub_Type values — add to SUB_TYPE_MAPPING if needed:")
        for val, cnt in other_vals.items():
            print(f"    ({cnt:>3}x)  '{val}'")

    # ── Class balance check ───────────────────────────────────────────────────
    present = set(dist.index.tolist())
    missing = set(SUB_TYPE_GROUPS) - {'OTHER'} - present
    if missing:
        print(f"\n  [WARN] These groups have zero rows: {missing}")

    min_n   = dist[dist.index != 'OTHER'].min() if len(dist) > 1 else dist.min()
    max_n   = dist.max()
    ratio   = max_n / min_n if min_n > 0 else float('inf')
    print(f"\n  Class imbalance ratio (max/min): {ratio:.1f}x")
    if ratio > 10:
        print(f"  SEVERE imbalance — use class_weight='balanced' or oversample.")
    elif ratio > 5:
        print(f"  Moderate imbalance — consider class_weight='balanced'.")
    else:
        print(f"  Imbalance acceptable.")

    return df


# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3 — X / y SEPARATION FOR SUB-TYPE CLASSIFICATION
#
# Target  → y = df["Sub_Type_Group"]   (8 broad groups)
# Features→ X = df[["Case_Text", "Case_Type", "Sub_Type_Group", "Court", "Year"]]
#
# Wait — Sub_Type_Group is BOTH the target y AND a feature column?
# No. When training the SUB-TYPE classifier, Sub_Type_Group IS y.
# For downstream tasks (e.g. verdict prediction), Sub_Type_Group becomes
# an input feature in X. This function serves the sub-type classifier only.
# ═══════════════════════════════════════════════════════════════════════════════

# Features for sub-type classification (Sub_Type_Group is the target, not a feature here)
SUB_TYPE_FEATURES = ['Case_Text', 'Case_Type', 'Court', 'Year']


def split_X_y_subtype(
    df: pd.DataFrame,
    label_col: str = 'Sub_Type_Group',
) -> Tuple[pd.DataFrame, pd.Series, dict]:
    """
    Separate the DataFrame into X (features) and y (sub-type target).

    X = df[["Case_Text", "Case_Type", "Court", "Year"]]
    y = df["Sub_Type_Group"]

    Note: Sub_Type (raw) is excluded from X — it is the source of y and
    would cause direct leakage. Sub_Type_Group is the target, not a feature.

    Parameters
    ----------
    df : pd.DataFrame
        Output of apply_sub_type_mapping().
    label_col : str
        Target column name. Default 'Sub_Type_Group'.

    Returns
    -------
    X : pd.DataFrame
        Feature matrix — 4 columns, leakage-free.
    y : pd.Series
        8-class sub-type label series.
    audit : dict
        Shape, columns, y distribution.

    Raises
    ------
    KeyError
        If label_col or any required feature column is missing.
    ValueError
        If any leakage column survives in X.
    """

    # ── Guard: label column must exist ───────────────────────────────────────
    if label_col not in df.columns:
        raise KeyError(
            f"'{label_col}' not found. "
            f"Did you run apply_sub_type_mapping() first?"
        )

    # ── Guard: all feature columns must exist ────────────────────────────────
    missing = [c for c in SUB_TYPE_FEATURES if c not in df.columns]
    if missing:
        raise KeyError(
            f"Missing feature columns: {missing}\n"
            f"Available: {list(df.columns)}"
        )

    # ── y: extract 8-class sub-type target ───────────────────────────────────
    y = df[label_col].copy()

    # ── X: 4 feature columns only ────────────────────────────────────────────
    # Sub_Type (raw) is excluded — it is the origin of y and would leak the label
    X = df[SUB_TYPE_FEATURES].copy()

    # ── Hard leakage assertion ────────────────────────────────────────────────
    leaked = [c for c in LEAKAGE_COLUMNS if c in X.columns]
    if leaked:
        raise ValueError(
            f"\n{'='*60}\n"
            f"LEAKAGE DETECTED in X: {leaked}\n"
            f"{'='*60}"
        )

    # ── Audit ─────────────────────────────────────────────────────────────────
    audit = {
        'X_shape':        X.shape,
        'X_columns':      list(X.columns),
        'y_name':         label_col,
        'y_distribution': y.value_counts().to_dict(),
        'rows_retained':  len(X),
        'leakage_free':   True,
    }

    print(f"\n{'='*55}")
    print(f"  X / y SEPARATION — SUB-TYPE CLASSIFIER")
    print(f"{'='*55}")
    print(f"  X shape    : {X.shape}")
    print(f"  X columns  : {list(X.columns)}")
    print(f"  y column   : {label_col}")
    print(f"  y shape    : {y.shape}")
    print(f"  Leakage    : PASSED — Sub_Type not in X")
    print(f"\n  y distribution:")
    for grp in SUB_TYPE_GROUPS:
        n = audit['y_distribution'].get(grp, 0)
        print(f"    {grp:<18}  {n:>5}")

    return X, y, audit


# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4 — FULL SUB-TYPE PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

def run_subtype_pipeline(
    df: pd.DataFrame,
    sub_type_col: str = 'Sub_Type',
) -> Tuple[pd.DataFrame, pd.Series, dict]:
    """
    One-call entry point for sub-type classification.

    Steps
    -----
    1. Map raw Sub_Type → broad Sub_Type_Group (8 classes).
    2. Report distribution and class balance.
    3. Separate into X (4 feature columns) and y (Sub_Type_Group).

    Parameters
    ----------
    df : pd.DataFrame
        Must contain Sub_Type, Case_Text, Case_Type, Court, Year.
    sub_type_col : str
        Raw sub-type column name. Default 'Sub_Type'.

    Returns
    -------
    X : pd.DataFrame
        Feature matrix — 4 columns, leakage-free.
    y : pd.Series
        8-class sub-type label series.
    audit : dict
        Shape, columns, y distribution, leakage status.
    """

    # Step 1: merge 36 raw sub-types into 8 broad groups
    print("Step 1/2  Merging Sub_Type into 8 broad groups...")
    df_merged = apply_sub_type_mapping(df, sub_type_col=sub_type_col)

    # Step 2: separate X and y for sub-type classifier training
    print("\nStep 2/2  Separating X and y (leakage-safe)...")
    X, y, audit = split_X_y_subtype(df_merged)

    return X, y, audit


# ═══════════════════════════════════════════════════════════════════════════════
# NLP + ML PREPROCESSING RECOMMENDATIONS
# ─────────────────────────────────────────────────────────────────────────────
# The notes below are not executable code — they describe what to do next
# once X and y are ready from run_subtype_pipeline().
#
#  ┌─────────────────────────────────────────────────────────────────────────┐
#  │  FEATURE: Case_Text  (NLP)                                              │
#  │  → TF-IDF (sklearn):                                                    │
#  │      TfidfVectorizer(max_features=10000, ngram_range=(1,2),             │
#  │                      stop_words='english')                              │
#  │  → Or: fine-tuned BERT / Legal-BERT for best accuracy                  │
#  │                                                                         │
#  │  FEATURE: Case_Type, Court  (categorical)                               │
#  │  → OneHotEncoder(handle_unknown='ignore')                               │
#  │                                                                         │
#  │  FEATURE: Year  (numeric)                                               │
#  │  → StandardScaler() or leave as-is for tree models                     │
#  │                                                                         │
#  │  RECOMMENDED MODELS (8-class classification):                           │
#  │  1. Logistic Regression   — fast baseline, good with TF-IDF            │
#  │  2. XGBoost / LightGBM    — strong with mixed features                 │
#  │  3. Fine-tuned BERT       — best accuracy for text-heavy signal        │
#  │                                                                         │
#  │  CLASS IMBALANCE (PROCEDURAL ~50%, GENERAL_CIVIL ~0.2%):               │
#  │  → Always use class_weight='balanced' in sklearn models                 │
#  │  → Or use SMOTE on the encoded feature matrix                          │
#  │  → Evaluate with macro-F1, not accuracy                                │
#  └─────────────────────────────────────────────────────────────────────────┘
# ═══════════════════════════════════════════════════════════════════════════════


# ═══════════════════════════════════════════════════════════════════════════════
# DEMO — runs when executed directly (no real CSV needed)
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':

    # Simulate a DataFrame with all 36 raw Sub_Type values
    raw_subtypes = list(SUB_TYPE_MAPPING.keys())  # all 36 known values

    # Repeat to get a reasonable demo size
    import math
    reps = math.ceil(100 / len(raw_subtypes))
    all_subtypes = (raw_subtypes * reps)[:100]

    sample_data = {
        'Case_Text':  [f"Full case text for row {i}" for i in range(100)],
        'Case_Type':  (['CIVIL'] * 25 + ['CRIMINAL'] * 25 +
                       ['CONTRACT'] * 25 + ['PROPERTY'] * 25),
        'Sub_Type':   all_subtypes,
        'Court':      (['Supreme Court'] * 50 + ['Circuit Court'] * 50),
        'Year':       list(range(1828, 1928)),
    }

    df_sample = pd.DataFrame(sample_data)

    print("Running sub-type classification pipeline on sample DataFrame...")
    X, y, audit = run_subtype_pipeline(df_sample, sub_type_col='Sub_Type')

    # ── Final confirmation ────────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  FINAL CONFIRMATION")
    print(f"{'='*55}")
    print(f"  X columns          : {list(X.columns)}")
    print(f"  X shape            : {X.shape}")
    print(f"  y name             : {y.name}")
    print(f"  y unique groups    : {sorted(y.unique())}")
    print(f"  Sub_Type in X      : {'Sub_Type' in X.columns}  (must be False)")
    print(f"  Sub_Type_Group in X: {'Sub_Type_Group' in X.columns}  (must be False)")
    print(f"\n  X and y are ready for NLP + ML sub-type classification.")

Running sub-type classification pipeline on sample DataFrame...
Step 1/2  Merging Sub_Type into 8 broad groups...

  SUB-TYPE GROUP DISTRIBUTION AFTER MERGING  (n=100)
  PROCEDURAL             21   21.0%  ██████████
  GENERAL_CRIMINAL        6    6.0%  ███
  DEBT_CONTRACT          15   15.0%  ███████
  PROPERTY_CIVIL         18   18.0%  █████████
  DEFAMATION_TORT        14   14.0%  ███████
  VIOLENT_CRIME          15   15.0%  ███████
  PROPERTY_CRIME          9    9.0%  ████
  GENERAL_CIVIL           2    2.0%  █
  OTHER                   0    0.0%  
  ──────────────────────────────────────────────────
  TOTAL                 100  100.0%

  Class imbalance ratio (max/min): 10.5x
  SEVERE imbalance — use class_weight='balanced' or oversample.

Step 2/2  Separating X and y (leakage-safe)...

  X / y SEPARATION — SUB-TYPE CLASSIFIER
  X shape    : (100, 4)
  X columns  : ['Case_Text', 'Case_Type', 'Court', 'Year']
  y column   : Sub_Type_Group
  y shape    : (100,)
  Leakage    : PASSED 

In [7]:
"""
Sub-Type Classification Pipeline — Fixed Version
==================================================
Fixes applied vs previous version:
  FIX 1 — GENERAL_CIVIL merged into PROCEDURAL (only 8 rows, unlearnable)
  FIX 2 — Full sklearn Pipeline: TF-IDF + OneHotEncoder + Logistic Regression
  FIX 3 — Stratified train/test split (mandatory for rare classes)
  FIX 4 — Evaluation with macro-F1 + full classification_report
  FIX 5 — class_weight='balanced' on all models (handles 175x imbalance)
  FIX 6 — Real data entry point: load your CSV, run one function
  FIX 7 — GENERAL_CIVIL removed: now 7 clean ML-ready groups

Final 7 groups:
    PROCEDURAL       — Appeal, Demurrer, writs, General Civil Procedure (merged in)
    GENERAL_CRIMINAL — General Criminal, Indictment
    VIOLENT_CRIME    — Homicide, Assault & Battery, Riot, Affray, Burglary/Robbery
    PROPERTY_CRIME   — Larceny, Counterfeiting, Perjury
    DEBT_CONTRACT    — Promissory Note, Bond, Debt Collection, Usury, General Contract
    PROPERTY_CIVIL   — Ejectment, Title Dispute, Mortgage Foreclosure, Fieri Facias, etc.
    DEFAMATION_TORT  — Slander, Defamation, Libel, Trespass, Negligence, etc.
    OTHER            — safe fallback for any unmapped value — never NaN
"""

import pandas as pd
import numpy as np
from typing import Tuple, Dict, List

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score


# ═══════════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ═══════════════════════════════════════════════════════════════════════════════

# FIX 1: GENERAL_CIVIL removed — merged into PROCEDURAL (was only 8 rows)
# 7 groups instead of 8 — all have enough rows to learn from
SUB_TYPE_GROUPS = [
    'PROCEDURAL',       # ~50%  — includes former GENERAL_CIVIL rows
    'GENERAL_CRIMINAL', # ~21%
    'DEBT_CONTRACT',    # ~10%
    'PROPERTY_CIVIL',   # ~7%
    'VIOLENT_CRIME',    # ~5%
    'DEFAMATION_TORT',  # ~4%
    'PROPERTY_CRIME',   # ~2.5%
    'OTHER',            # safe fallback
]

# Leakage columns — must never appear in X
LEAKAGE_COLUMNS = [
    'verdict_text', 'verdict_clean', 'keyword',
    'verdict_category', 'outcome_simple', 'four_class_label',
    'Sub_Type',        # raw source of y — direct leakage
    'Sub_Type_Group',  # the target itself
]

# FIX 2: separate feature types for correct encoding in ColumnTransformer
TEXT_FEATURE = 'Case_Text'                   # NLP → TF-IDF
CAT_FEATURES = ['Case_Type', 'Court']        # categorical → OneHotEncoder
NUM_FEATURES = ['Year']                      # numeric → passthrough
ALL_FEATURES = [TEXT_FEATURE] + CAT_FEATURES + NUM_FEATURES


# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1 — SUB-TYPE MERGING MAP
# 36 raw Sub_Type values → 7 broad groups
# FIX 1: 'General Civil Procedure' now maps to PROCEDURAL (was GENERAL_CIVIL)
# ═══════════════════════════════════════════════════════════════════════════════

SUB_TYPE_MAPPING: Dict[str, str] = {

    # ── PROCEDURAL (~50%) ─────────────────────────────────────────────────────
    'Appeal':                   'PROCEDURAL',
    'Demurrer':                 'PROCEDURAL',
    'Pleading':                 'PROCEDURAL',
    'Writ':                     'PROCEDURAL',
    'Capias':                   'PROCEDURAL',
    'Scire Facias':             'PROCEDURAL',
    'Attachment':               'PROCEDURAL',
    'General Civil Procedure':  'PROCEDURAL',  # FIX 1: moved from GENERAL_CIVIL

    # ── GENERAL_CRIMINAL (~21%) ───────────────────────────────────────────────
    'General Criminal':         'GENERAL_CRIMINAL',
    'Indictment':               'GENERAL_CRIMINAL',

    # ── VIOLENT_CRIME (~5.4%) ─────────────────────────────────────────────────
    'Homicide':                 'VIOLENT_CRIME',
    'Assault & Battery':        'VIOLENT_CRIME',
    'Riot':                     'VIOLENT_CRIME',
    'Affray':                   'VIOLENT_CRIME',
    'Burglary/Robbery':         'VIOLENT_CRIME',

    # ── PROPERTY_CRIME (~2.5%) ────────────────────────────────────────────────
    'Larceny':                  'PROPERTY_CRIME',
    'Counterfeiting':           'PROPERTY_CRIME',
    'Perjury':                  'PROPERTY_CRIME',

    # ── DEBT_CONTRACT (~10.3%) ────────────────────────────────────────────────
    'Promissory Note':          'DEBT_CONTRACT',
    'Bond':                     'DEBT_CONTRACT',
    'Debt Collection':          'DEBT_CONTRACT',
    'Usury':                    'DEBT_CONTRACT',
    'General Contract':         'DEBT_CONTRACT',

    # ── PROPERTY_CIVIL (~7.1%) ────────────────────────────────────────────────
    'Ejectment':                'PROPERTY_CIVIL',
    'Title Dispute':            'PROPERTY_CIVIL',
    'Mortgage Foreclosure':     'PROPERTY_CIVIL',
    'General Property':         'PROPERTY_CIVIL',
    'Fieri Facias':             'PROPERTY_CIVIL',
    'General Execution':        'PROPERTY_CIVIL',

    # ── DEFAMATION_TORT (~4.2%) ───────────────────────────────────────────────
    'Slander':                  'DEFAMATION_TORT',
    'Defamation':               'DEFAMATION_TORT',
    'Libel':                    'DEFAMATION_TORT',
    'Trespass':                 'DEFAMATION_TORT',
    'Negligence':               'DEFAMATION_TORT',
    'Conversion':               'DEFAMATION_TORT',
    'General Torts':            'DEFAMATION_TORT',
}


# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2 — APPLY MAPPING
# ═══════════════════════════════════════════════════════════════════════════════

def apply_sub_type_mapping(
    df: pd.DataFrame,
    sub_type_col: str = 'Sub_Type',
) -> pd.DataFrame:
    """
    Map raw Sub_Type → broad Sub_Type_Group.
    Unmapped values → 'OTHER' (never NaN).
    Adds 'Sub_Type_Group' column. Original Sub_Type kept for audit only.
    """
    if sub_type_col not in df.columns:
        raise KeyError(f"Column '{sub_type_col}' not found. Available: {list(df.columns)}")

    df = df.copy()
    df['Sub_Type_Group'] = df[sub_type_col].map(SUB_TYPE_MAPPING).fillna('OTHER')

    if df['Sub_Type_Group'].isna().sum() > 0:
        raise ValueError("Sub_Type_Group has NaN values — check fillna logic.")

    dist  = df['Sub_Type_Group'].value_counts()
    total = len(df)

    print(f"\n{'='*55}")
    print(f"  SUB-TYPE GROUP DISTRIBUTION  (n={total:,})")
    print(f"{'='*55}")
    for grp in SUB_TYPE_GROUPS:
        n   = dist.get(grp, 0)
        pct = n / total * 100
        bar = '█' * int(pct / 2)
        print(f"  {grp:<18}  {n:>5}  {pct:5.1f}%  {bar}")
    print(f"  {'─'*50}")
    print(f"  {'TOTAL':<18}  {total:>5}  100.0%")

    other_n = dist.get('OTHER', 0)
    if other_n > 0:
        unmapped = df.loc[df['Sub_Type_Group'] == 'OTHER', sub_type_col].value_counts()
        print(f"\n  [WARN] {other_n} rows → OTHER. Unmapped values:")
        for val, cnt in unmapped.items():
            print(f"    ({cnt:>3}x)  '{val}'  ← add to SUB_TYPE_MAPPING if needed")

    counts   = dist[dist.index != 'OTHER']
    ratio    = counts.max() / counts.min() if counts.min() > 0 else float('inf')
    severity = 'SEVERE' if ratio > 50 else 'HIGH' if ratio > 10 else 'moderate'
    print(f"\n  Imbalance ratio (max/min, excl. OTHER): {ratio:.1f}x  [{severity}]")
    if ratio > 10:
        print(f"  → Use class_weight='balanced' (already set in build_ml_pipeline).")
        print(f"  → Evaluate with macro-F1, NOT accuracy.")

    return df


# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3 — X / y SEPARATION
# ═══════════════════════════════════════════════════════════════════════════════

def split_X_y_subtype(
    df: pd.DataFrame,
    label_col: str = 'Sub_Type_Group',
) -> Tuple[pd.DataFrame, pd.Series, dict]:
    """
    X = df[["Case_Text", "Case_Type", "Court", "Year"]]
    y = df["Sub_Type_Group"]

    Sub_Type (raw) excluded from X — it is the source of y.
    """
    if label_col not in df.columns:
        raise KeyError(f"'{label_col}' not found. Run apply_sub_type_mapping() first.")

    missing = [c for c in ALL_FEATURES if c not in df.columns]
    if missing:
        raise KeyError(f"Missing feature columns: {missing}")

    y = df[label_col].copy()
    X = df[ALL_FEATURES].copy()

    leaked = [c for c in LEAKAGE_COLUMNS if c in X.columns]
    if leaked:
        raise ValueError(f"LEAKAGE DETECTED in X: {leaked}")

    audit = {
        'X_shape':        X.shape,
        'X_columns':      list(X.columns),
        'y_distribution': y.value_counts().to_dict(),
        'leakage_free':   True,
    }

    print(f"\n{'='*55}")
    print(f"  X / y SEPARATION — SUB-TYPE CLASSIFIER")
    print(f"{'='*55}")
    print(f"  X shape    : {X.shape}")
    print(f"  X columns  : {list(X.columns)}")
    print(f"  y column   : {label_col}  ({y.nunique()} classes)")
    print(f"  Leakage    : PASSED")

    return X, y, audit


# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4 — BUILD ML PIPELINE
# FIX 2: TF-IDF on text + OneHotEncoder on categoricals + Logistic Regression
# FIX 5: class_weight='balanced' handles severe imbalance automatically
# ═══════════════════════════════════════════════════════════════════════════════

def build_ml_pipeline(
    max_tfidf_features: int = 10000,
    ngram_range: tuple = (1, 2),
    C: float = 1.0,
) -> Pipeline:
    """
    Full sklearn Pipeline: preprocessing → classifier.

    Case_Text  → TfidfVectorizer  (NLP, unigrams + bigrams)
    Case_Type  → OneHotEncoder    (categorical)
    Court      → OneHotEncoder    (categorical)
    Year       → passthrough      (numeric)
    Classifier → LogisticRegression(class_weight='balanced')
    """
    preprocessor = ColumnTransformer(
        transformers=[
            # NLP: TF-IDF on Case_Text
            ('tfidf', TfidfVectorizer(
                max_features=max_tfidf_features,
                ngram_range=ngram_range,
                sublinear_tf=True,   # dampens very common terms
                min_df=2,            # ignore terms in < 2 documents
                strip_accents='unicode',
                analyzer='word',
            ), TEXT_FEATURE),

            # Categorical: one-hot encode Case_Type and Court
            # handle_unknown='ignore' zeros out unseen values at predict time
            ('cat', OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False,
            ), CAT_FEATURES),

            # Numeric: Year passes through unchanged
            ('num', 'passthrough', NUM_FEATURES),
        ],
        remainder='drop',  # drop any extra columns silently
    )

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        # FIX 5: class_weight='balanced' upweights minority classes automatically
        ('classifier', LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            C=C,
            solver='lbfgs',
            multi_class='multinomial',
            random_state=42,
        )),
    ])

    return pipeline


# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 5 — STRATIFIED SPLIT + TRAIN + EVALUATE
# FIX 3: stratify=y — mandatory for rare classes (PROPERTY_CRIME ~100 rows)
# FIX 4: macro-F1 + classification_report — fair across all 7 classes
# ═══════════════════════════════════════════════════════════════════════════════

def train_and_evaluate(
    X: pd.DataFrame,
    y: pd.Series,
    test_size: float = 0.2,
    random_state: int = 42,
) -> dict:
    """
    Stratified split → fit → evaluate with macro-F1.

    FIX 3: stratify=y preserves rare class proportions in both halves.
    FIX 4: macro-F1 weights all 7 classes equally.
           A model always predicting PROCEDURAL scores ~0.07 macro-F1
           despite 50% accuracy — this exposes the imbalance problem clearly.
    """

    # FIX 3: stratified split — mandatory
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )

    # Class coverage check
    missing_train = set(y.unique()) - set(y_train.unique())
    missing_test  = set(y.unique()) - set(y_test.unique())
    if missing_train:
        print(f"  [WARN] Classes absent from train: {missing_train}")
    if missing_test:
        print(f"  [WARN] Classes absent from test: {missing_test}")

    print(f"\n{'='*55}")
    print(f"  TRAIN / TEST SPLIT")
    print(f"{'='*55}")
    print(f"  Train: {len(X_train):>5} rows   Test: {len(X_test):>5} rows")
    print(f"\n  {'Group':<18}  {'Train':>6}  {'Test':>6}")
    print(f"  {'─'*34}")
    for grp in SUB_TYPE_GROUPS:
        n_tr = int((y_train == grp).sum())
        n_te = int((y_test  == grp).sum())
        if n_tr > 0 or n_te > 0:
            print(f"  {grp:<18}  {n_tr:>6}  {n_te:>6}")

    # Train
    print(f"\n  Training pipeline...")
    pipeline = build_ml_pipeline()
    pipeline.fit(X_train, y_train)

    # FIX 4: evaluate with macro-F1
    y_pred   = pipeline.predict(X_test)
    macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    print(f"\n{'='*55}")
    print(f"  EVALUATION RESULTS  (macro-F1 = {macro_f1:.4f})")
    print(f"{'='*55}")
    print(f"  macro-F1 weights all {y.nunique()} classes equally.\n")
    print(classification_report(
        y_test, y_pred,
        labels=[g for g in SUB_TYPE_GROUPS if g != 'OTHER'],
        zero_division=0,
    ))

    print(f"  Interpretation:")
    if macro_f1 >= 0.75:
        print(f"  macro-F1 {macro_f1:.2f} — good. All classes learned well.")
    elif macro_f1 >= 0.50:
        print(f"  macro-F1 {macro_f1:.2f} — acceptable baseline.")
        print(f"  Try: increase TF-IDF features, tune C, or try XGBoost.")
    else:
        print(f"  macro-F1 {macro_f1:.2f} — weak. Possible causes:")
        print(f"    1. Rare classes too few — try SMOTE oversampling")
        print(f"    2. Case_Text has low discriminating signal")
        print(f"    3. Try fine-tuned Legal-BERT for much better text encoding")

    return {
        'pipeline': pipeline,
        'X_train':  X_train, 'X_test':  X_test,
        'y_train':  y_train, 'y_test':  y_test,
        'y_pred':   y_pred,  'macro_f1': macro_f1,
    }


# ═══════════════════════════════════════════════════════════════════════════════
# FULL PIPELINE — single call entry point
# ═══════════════════════════════════════════════════════════════════════════════

def run_subtype_pipeline(
    df: pd.DataFrame,
    sub_type_col: str = 'Sub_Type',
    train: bool = True,
    test_size: float = 0.2,
) -> dict:
    """
    One-call entry point for sub-type classification.

    FIX 6 — load your real data:
        df = pd.read_csv('your_cases.csv')
        results = run_subtype_pipeline(df)

    Required CSV columns: Sub_Type, Case_Text, Case_Type, Court, Year
    """
    print("Step 1/3  Merging Sub_Type into 7 broad groups...")
    df_merged = apply_sub_type_mapping(df, sub_type_col=sub_type_col)

    print("\nStep 2/3  Separating X and y...")
    X, y, audit = split_X_y_subtype(df_merged)

    results = {'X': X, 'y': y, 'audit': audit}

    if train:
        print("\nStep 3/3  Training and evaluating ML pipeline...")
        results.update(train_and_evaluate(X, y, test_size=test_size))
    else:
        print("\nStep 3/3  Skipped (train=False). X and y ready for external model.")

    return results


# ═══════════════════════════════════════════════════════════════════════════════
# DEMO
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':

    # ── TO USE YOUR REAL DATA — replace the demo block below with: ────────────
    # df_real = pd.read_csv('your_legal_cases.csv')
    # results = run_subtype_pipeline(df_real)
    # ─────────────────────────────────────────────────────────────────────────

    # Demo with realistic distribution proportional to your actual data
    realistic_counts = {
        'Appeal': 140, 'General Criminal': 82, 'Demurrer': 56,
        'Promissory Note': 25, 'Bond': 11, 'Ejectment': 10,
        'Slander': 9,  'Title Dispute': 9,  'Larceny': 7,
        'Homicide': 7, 'Mortgage Foreclosure': 7, 'Assault & Battery': 5,
        'Defamation': 5, 'Indictment': 5, 'General Civil Procedure': 1,
        'Fieri Facias': 3, 'Debt Collection': 2, 'General Contract': 2,
        'Perjury': 2, 'General Execution': 2, 'Trespass': 2,
        'Libel': 1, 'Riot': 1, 'Counterfeiting': 1, 'General Torts': 1,
    }

    criminal_types = {'General Criminal', 'Larceny', 'Homicide',
                      'Assault & Battery', 'Indictment', 'Perjury',
                      'Counterfeiting', 'Riot', 'Affray'}
    contract_types = {'Promissory Note', 'Bond', 'Debt Collection',
                      'Usury', 'General Contract'}

    rows = []
    for sub_type, count in realistic_counts.items():
        for i in range(count):
            case_type = (
                'CRIMINAL' if sub_type in criminal_types else
                'CONTRACT' if sub_type in contract_types else
                'CIVIL'    if sub_type in {'Appeal', 'Demurrer', 'General Civil Procedure'} else
                'PROPERTY'
            )
            rows.append({
                'Sub_Type':  sub_type,
                'Case_Text': (f"The {sub_type.lower()} matter came before the court. "
                              f"Defendant argued the judgment was in error. "
                              f"The court reviewed the record and testimony. Case {i}."),
                'Case_Type': case_type,
                'Court':     'Supreme Court' if i % 2 == 0 else 'Circuit Court',
                'Year':      1828 + (i % 40),
            })

    df_demo = pd.DataFrame(rows)
    print(f"Demo: {len(df_demo)} rows, {df_demo['Sub_Type'].nunique()} unique Sub_Types\n")

    results = run_subtype_pipeline(df_demo, train=True)

    print(f"\n{'='*55}")
    print(f"  FINAL CONFIRMATION")
    print(f"{'='*55}")
    X, y = results['X'], results['y']
    print(f"  X columns          : {list(X.columns)}")
    print(f"  X shape            : {X.shape}")
    print(f"  y unique groups    : {sorted(y.unique())}")
    print(f"  Sub_Type in X      : {'Sub_Type' in X.columns}  (must be False)")
    print(f"  Sub_Type_Group in X: {'Sub_Type_Group' in X.columns}  (must be False)")
    if 'macro_f1' in results:
        print(f"  Macro-F1 score     : {results['macro_f1']:.4f}")
    print(f"\n  Pipeline ready for NLP + ML sub-type classification.")

Demo: 396 rows, 25 unique Sub_Types

Step 1/3  Merging Sub_Type into 7 broad groups...

  SUB-TYPE GROUP DISTRIBUTION  (n=396)
  PROCEDURAL            197   49.7%  ████████████████████████
  GENERAL_CRIMINAL       87   22.0%  ██████████
  DEBT_CONTRACT          40   10.1%  █████
  PROPERTY_CIVIL         31    7.8%  ███
  VIOLENT_CRIME          13    3.3%  █
  DEFAMATION_TORT        18    4.5%  ██
  PROPERTY_CRIME         10    2.5%  █
  OTHER                   0    0.0%  
  ──────────────────────────────────────────────────
  TOTAL                 396  100.0%

  Imbalance ratio (max/min, excl. OTHER): 19.7x  [HIGH]
  → Use class_weight='balanced' (already set in build_ml_pipeline).
  → Evaluate with macro-F1, NOT accuracy.

Step 2/3  Separating X and y...

  X / y SEPARATION — SUB-TYPE CLASSIFIER
  X shape    : (396, 4)
  X columns  : ['Case_Text', 'Case_Type', 'Court', 'Year']
  y column   : Sub_Type_Group  (7 classes)
  Leakage    : PASSED

Step 3/3  Training and evaluating ML pipeli

c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



  EVALUATION RESULTS  (macro-F1 = 0.9159)
  macro-F1 weights all 7 classes equally.

                  precision    recall  f1-score   support

      PROCEDURAL       1.00      1.00      1.00        40
GENERAL_CRIMINAL       0.89      1.00      0.94        17
   DEBT_CONTRACT       1.00      1.00      1.00         8
  PROPERTY_CIVIL       1.00      1.00      1.00         6
   VIOLENT_CRIME       1.00      0.67      0.80         3
 DEFAMATION_TORT       1.00      1.00      1.00         4
  PROPERTY_CRIME       1.00      0.50      0.67         2

        accuracy                           0.97        80
       macro avg       0.98      0.88      0.92        80
    weighted avg       0.98      0.97      0.97        80

  Interpretation:
  macro-F1 0.92 — good. All classes learned well.

  FINAL CONFIRMATION
  X columns          : ['Case_Text', 'Case_Type', 'Court', 'Year']
  X shape            : (396, 4)
  y unique groups    : ['DEBT_CONTRACT', 'DEFAMATION_TORT', 'GENERAL_CRIMINAL', 'PRO

c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [8]:
def map_final_subtype(st):
    st = str(st).lower().strip()

    # 1. APPELLATE PROCEDURE (appeal vocabulary)
    if "appeal" in st:
        return "APPELLATE_PROCEDURE"

    # 2. TRIAL PROCEDURE (pleading / writ vocabulary)
    if st in ["demurrer", "pleading", "writ", "capias", "scire facias", "general civil procedure"]:
        return "TRIAL_PROCEDURE"

    # 3. CONTRACT / DEBT (financial obligation vocabulary)
    if st in ["promissory note", "bond", "debt collection", "usury", "general contract"]:
        return "CONTRACT_DEBT"

    # 4. PROPERTY RIGHTS (ownership / land vocabulary)
    if st in ["ejectment", "title dispute", "general property", "mortgage foreclosure"]:
        return "PROPERTY_RIGHTS"

    # 5. EXECUTION / ENFORCEMENT (judgment enforcement vocabulary)
    if st in ["fieri facias", "general execution", "attachment"]:
        return "EXECUTION_ENFORCEMENT"

    # 6. DEFAMATION (reputation vocabulary)
    if st in ["slander", "libel", "defamation"]:
        return "DEFAMATION"

    # 7. GENERAL TORT (injury / damage vocabulary)
    if st in ["trespass", "negligence", "conversion", "general torts"]:
        return "GENERAL_TORT"

    # 8. CRIMINAL OFFENSE (all crimes merged)
    if st in ["general criminal", "larceny", "homicide", "assault & battery",
              "indictment", "perjury", "riot", "counterfeiting",
              "burglary/robbery", "affray"]:
        return "CRIMINAL_OFFENSE"

    return "OTHER"

In [9]:
df["Final_SubType_Label"] = df["Sub_Type"].apply(map_final_subtype)
dist_counts = df["Final_SubType_Label"].value_counts()
dist_percent = df["Final_SubType_Label"].value_counts(normalize=True) * 100

distribution = (
    pd.DataFrame({
        "Count": dist_counts,
        "Percentage (%)": dist_percent.round(2)
    })
    .sort_values("Count", ascending=False)
)

print(distribution)

                       Count  Percentage (%)
Final_SubType_Label                         
CRIMINAL_OFFENSE        4160           25.85
APPELLATE_PROCEDURE     3614           22.46
CONTRACT_DEBT           2402           14.93
PROPERTY_RIGHTS         1880           11.68
GENERAL_TORT            1834           11.40
TRIAL_PROCEDURE         1729           10.75
DEFAMATION               221            1.37
EXECUTION_ENFORCEMENT    199            1.24
OTHER                     52            0.32


In [10]:
"""
Civil & Criminal Hierarchical Classification Pipeline
======================================================

ARCHITECTURE
────────────
  L0  Case_Type classifier  — 5 classes: CIVIL, CRIMINAL, CONTRACT, PROPERTY, TORTS
  L1  Binary classifier     — CIVIL vs CRIMINAL  (routes to L2)
  L2a Civil sub-classifier  — APPELLATE vs TRIAL_ENFORCEMENT
  L2b Criminal sub-classifier — GENERAL_CRIMINAL vs SPECIFIC_OFFENSE

  CONTRACT / PROPERTY / TORTS: L0 prediction is final (no L2 — insufficient data)

LIMITATIONS FIXED vs PREVIOUS VERSION
──────────────────────────────────────
  FIX 1  — Case_Type (L0) added as a proper 5-class classifier
  FIX 2  — Leakage guard: Sub_Type and all label columns blocked from X
  FIX 3  — Demo text no longer embeds Sub_Type name → leakage-free synthetic data
  FIX 4  — Confidence gating at every level (threshold=0.45) → OTHER fallback
  FIX 5  — L1→L2 hierarchy enforced at predict time (impossible label combos blocked)
  FIX 6  — 5-fold stratified cross-validation at every level (not just holdout)
  FIX 7  — class_weight='balanced' on all classifiers
  FIX 8  — OTHER excluded from training; confidence-based assignment at predict time
  FIX 9  — Rare-class warnings printed if any class < MIN_CLASS_ROWS
  FIX 10 — predict_batch returns full confidence scores + hierarchy validity flag
  FIX 11 — Confusion matrices printed at every level
  FIX 12 — Macro-F1 used everywhere (not accuracy)
"""

import pandas as pd
import numpy as np
import warnings
from typing import Dict, Tuple, Optional, List

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, f1_score, confusion_matrix

warnings.filterwarnings('ignore')


# ═══════════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ═══════════════════════════════════════════════════════════════════════════════

TEXT_FEATURE  = 'Case_Text'
CAT_FEATURES  = ['Court']
NUM_FEATURES  = ['Year']
ALL_FEATURES  = [TEXT_FEATURE] + CAT_FEATURES + NUM_FEATURES

CONFIDENCE_THRESHOLD = 0.45   # below this → 'OTHER' at any level
MIN_CLASS_ROWS       = 30     # warn if any class in training falls below this
TEST_SIZE            = 0.20
CV_FOLDS             = 5
RANDOM_STATE         = 42

# Columns that must never appear in X (leakage guard)
LEAKAGE_COLUMNS = [
    'Sub_Type', 'Sub_Type_Group',
    'L0_Label', 'L1_Label', 'L2_Civil', 'L2_Criminal',
    'verdict_text', 'verdict_clean', 'keyword',
    'verdict_category', 'outcome_simple', 'four_class_label',
]


# ═══════════════════════════════════════════════════════════════════════════════
# MAPPING TABLES
# ═══════════════════════════════════════════════════════════════════════════════

# ── L0: Sub_Type → Case_Type (5 broad groups) ──────────────────────────────
L0_MAPPING: Dict[str, str] = {
    # CIVIL
    'Appeal':                  'CIVIL',
    'Demurrer':                'CIVIL',
    'Fieri Facias':            'CIVIL',
    'General Execution':       'CIVIL',
    'General Civil Procedure': 'CIVIL',
    'Pleading':                'CIVIL',
    'Capias':                  'CIVIL',
    'Writ':                    'CIVIL',
    'Attachment':              'CIVIL',
    'Scire Facias':            'CIVIL',
    # CRIMINAL
    'General Criminal':        'CRIMINAL',
    'Indictment':              'CRIMINAL',
    'Homicide':                'CRIMINAL',
    'Assault & Battery':       'CRIMINAL',
    'Larceny':                 'CRIMINAL',
    'Perjury':                 'CRIMINAL',
    'Riot':                    'CRIMINAL',
    'Counterfeiting':          'CRIMINAL',
    'Burglary/Robbery':        'CRIMINAL',
    'Affray':                  'CRIMINAL',
    # CONTRACT
    'Promissory Note':         'CONTRACT',
    'Bond':                    'CONTRACT',
    'Debt Collection':         'CONTRACT',
    'General Contract':        'CONTRACT',
    'Usury':                   'CONTRACT',
    # PROPERTY
    'Ejectment':               'PROPERTY',
    'Title Dispute':           'PROPERTY',
    'Mortgage Foreclosure':    'PROPERTY',
    'General Property':        'PROPERTY',
    # TORTS
    'Slander':                 'TORTS',
    'Defamation':              'TORTS',
    'Libel':                   'TORTS',
    'Trespass':                'TORTS',
    'General Torts':           'TORTS',
    'Negligence':              'TORTS',
    'Conversion':              'TORTS',
}

# ── L1: same as L0 but only CIVIL/CRIMINAL rows used ──────────────────────
L1_MAPPING: Dict[str, str] = {
    k: v for k, v in L0_MAPPING.items() if v in ('CIVIL', 'CRIMINAL')
}

# ── L2 Civil ───────────────────────────────────────────────────────────────
L2_CIVIL_MAPPING: Dict[str, str] = {
    'Appeal':                  'APPELLATE',
    'Writ':                    'APPELLATE',
    'Scire Facias':            'APPELLATE',
    'Demurrer':                'TRIAL_ENFORCEMENT',
    'Fieri Facias':            'TRIAL_ENFORCEMENT',
    'General Execution':       'TRIAL_ENFORCEMENT',
    'General Civil Procedure': 'TRIAL_ENFORCEMENT',
    'Pleading':                'TRIAL_ENFORCEMENT',
    'Capias':                  'TRIAL_ENFORCEMENT',
    'Attachment':              'TRIAL_ENFORCEMENT',
}

# ── L2 Criminal ────────────────────────────────────────────────────────────
L2_CRIMINAL_MAPPING: Dict[str, str] = {
    'General Criminal':  'GENERAL_CRIMINAL',
    'Indictment':        'GENERAL_CRIMINAL',
    'Homicide':          'SPECIFIC_OFFENSE',
    'Assault & Battery': 'SPECIFIC_OFFENSE',
    'Larceny':           'SPECIFIC_OFFENSE',
    'Perjury':           'SPECIFIC_OFFENSE',
    'Riot':              'SPECIFIC_OFFENSE',
    'Counterfeiting':    'SPECIFIC_OFFENSE',
    'Burglary/Robbery':  'SPECIFIC_OFFENSE',
    'Affray':            'SPECIFIC_OFFENSE',
}

# ── Hierarchy enforcement ──────────────────────────────────────────────────
L0_CLASSES         = ['CIVIL', 'CRIMINAL', 'CONTRACT', 'PROPERTY', 'TORTS']
L2_HAS_SUBCLASSIFIER = {'CIVIL', 'CRIMINAL'}     # only these get L2
L1_TO_L2_CLASSES   = {
    'CIVIL':    ['APPELLATE', 'TRIAL_ENFORCEMENT'],
    'CRIMINAL': ['GENERAL_CRIMINAL', 'SPECIFIC_OFFENSE'],
}


# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1 — APPLY ALL MAPPINGS
# ═══════════════════════════════════════════════════════════════════════════════

def apply_mappings(
    df: pd.DataFrame,
    sub_type_col: str = 'Sub_Type',
) -> pd.DataFrame:
    """
    Adds four label columns:
        L0_Label    — CIVIL / CRIMINAL / CONTRACT / PROPERTY / TORTS / OTHER
        L1_Label    — CIVIL / CRIMINAL / NaN  (only for civil+criminal rows)
        L2_Civil    — APPELLATE / TRIAL_ENFORCEMENT / NaN
        L2_Criminal — GENERAL_CRIMINAL / SPECIFIC_OFFENSE / NaN

    FIX 2: Sub_Type is the source of all labels — never touches X.
    FIX 8: OTHER rows exist in the data but are excluded from training.
    """
    if sub_type_col not in df.columns:
        raise KeyError(
            f"Column '{sub_type_col}' not found.\n"
            f"Available columns: {list(df.columns)}"
        )

    df = df.copy()
    df['L0_Label']    = df[sub_type_col].map(L0_MAPPING).fillna('OTHER')
    df['L1_Label']    = df[sub_type_col].map(L1_MAPPING)       # NaN if not civil/criminal
    df['L2_Civil']    = df[sub_type_col].map(L2_CIVIL_MAPPING)  # NaN if not civil
    df['L2_Criminal'] = df[sub_type_col].map(L2_CRIMINAL_MAPPING)  # NaN if not criminal

    total = len(df)
    print(f"\n{'═'*60}")
    print(f"  MAPPING REPORT  (n={total:,})")
    print(f"{'═'*60}")

    l0_dist = df['L0_Label'].value_counts()
    for l0 in L0_CLASSES + ['OTHER']:
        n   = l0_dist.get(l0, 0)
        pct = n / total * 100
        bar = '█' * max(1, int(pct / 3))
        print(f"\n  {l0:<12}  {n:>5} rows  ({pct:5.1f}%)  {bar}")

        subset = df[df['L0_Label'] == l0]
        if l0 == 'CIVIL' and len(subset):
            l2c = subset['L2_Civil'].value_counts()
            for cls, cnt in l2c.items():
                flag = '  ⚠ LOW — unreliable' if cnt < MIN_CLASS_ROWS else ''
                print(f"    ├── {cls:<22} {cnt:>5}{flag}")
        elif l0 == 'CRIMINAL' and len(subset):
            l2cr = subset['L2_Criminal'].value_counts()
            for cls, cnt in l2cr.items():
                flag = '  ⚠ LOW — unreliable' if cnt < MIN_CLASS_ROWS else ''
                print(f"    ├── {cls:<22} {cnt:>5}{flag}")

    # Imbalance check
    print(f"\n  IMBALANCE CHECK")
    for l0, col in [('CIVIL', 'L2_Civil'), ('CRIMINAL', 'L2_Criminal')]:
        subset = df[df['L0_Label'] == l0]
        if len(subset) == 0:
            continue
        counts = subset[col].value_counts().dropna()
        if len(counts) < 2:
            continue
        ratio = counts.max() / counts.min()
        sev   = 'SEVERE' if ratio > 50 else 'HIGH' if ratio > 10 else 'OK'
        print(f"  {l0:<10}  imbalance {ratio:>6.1f}x  [{sev}]")

    # OTHER warning
    n_other = l0_dist.get('OTHER', 0)
    if n_other > 0:
        unmapped = df.loc[df['L0_Label'] == 'OTHER', sub_type_col].value_counts()
        print(f"\n  [WARN] {n_other} rows → OTHER. Unmapped Sub_Types:")
        for val, cnt in unmapped.items():
            print(f"    ({cnt:>3}x)  '{val}'  ← add to L0_MAPPING if needed")

    return df


# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2 — LEAKAGE GUARD
# ═══════════════════════════════════════════════════════════════════════════════

def safe_X(df: pd.DataFrame) -> pd.DataFrame:
    """
    FIX 2: Returns feature-only DataFrame.
    Raises ValueError if any leakage column is present.
    """
    X = df[ALL_FEATURES].copy()
    leaked = [c for c in LEAKAGE_COLUMNS if c in X.columns]
    if leaked:
        raise ValueError(
            f"LEAKAGE DETECTED — these columns must never appear in X:\n  {leaked}"
        )
    return X


# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3 — PIPELINE FACTORY
# ═══════════════════════════════════════════════════════════════════════════════

def build_pipeline(C: float = 1.0, max_features: int = 10_000) -> Pipeline:
    """
    Shared sklearn Pipeline:
        Case_Text  → TF-IDF (unigrams + bigrams, sublinear_tf)
        Court      → OneHotEncoder
        Year       → passthrough
        Classifier → LogisticRegression(class_weight='balanced')

    FIX 7: class_weight='balanced' handles all imbalance levels automatically.
    """
    preprocessor = ColumnTransformer(
        transformers=[
            ('tfidf', TfidfVectorizer(
                max_features=max_features,
                ngram_range=(1, 2),
                sublinear_tf=True,
                min_df=2,
                strip_accents='unicode',
                analyzer='word',
            ), TEXT_FEATURE),
            ('cat', OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False,
            ), CAT_FEATURES),
            ('num', 'passthrough', NUM_FEATURES),
        ],
        remainder='drop',
    )

    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            C=C,
            solver='lbfgs',
            multi_class='multinomial',
            random_state=RANDOM_STATE,
        )),
    ])


# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4 — TRAIN ONE LEVEL
# ═══════════════════════════════════════════════════════════════════════════════

def train_level(
    df_subset: pd.DataFrame,
    label_col: str,
    level_name: str,
    test_size: float = TEST_SIZE,
    cv_folds: int = CV_FOLDS,
) -> dict:
    """
    Train and evaluate one classification level with:
      - Stratified train/test split
      - 5-fold stratified cross-validation  (FIX 6)
      - macro-F1 evaluation                  (FIX 12)
      - Confusion matrix                     (FIX 11)
      - Rare-class warnings                  (FIX 9)

    FIX 8: Rows with NaN labels (OTHER rows) are dropped before training.
    FIX 2: safe_X() blocks leakage columns.
    """
    df_clean = df_subset.dropna(subset=[label_col]).copy()
    if len(df_clean) == 0:
        raise ValueError(f"[{level_name}] No rows with valid labels for '{label_col}'.")

    missing_feats = [c for c in ALL_FEATURES if c not in df_clean.columns]
    if missing_feats:
        raise KeyError(f"[{level_name}] Missing feature columns: {missing_feats}")

    X = safe_X(df_clean)      # FIX 2: leakage guard
    y = df_clean[label_col]

    # FIX 9: rare-class warnings
    class_counts = y.value_counts()
    for cls, cnt in class_counts.items():
        if cnt < MIN_CLASS_ROWS:
            print(f"  ⚠  [{level_name}] '{cls}' has only {cnt} rows — "
                  f"F1 for this class will be unreliable.")

    # FIX 3: stratified split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    print(f"\n{'═'*60}")
    print(f"  {level_name}  —  TRAINING")
    print(f"{'═'*60}")
    print(f"  Train: {len(X_train):,}   Test: {len(X_test):,}")
    print(f"\n  {'Class':<26} {'Train':>6}  {'Test':>6}")
    print(f"  {'─'*40}")
    for cls in sorted(y.unique()):
        n_tr = int((y_train == cls).sum())
        n_te = int((y_test  == cls).sum())
        print(f"  {cls:<26} {n_tr:>6}  {n_te:>6}")

    pipeline = build_pipeline()
    pipeline.fit(X_train, y_train)

    # FIX 6: cross-validation on full labelled set
    cv     = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_STATE)
    cv_f1  = cross_val_score(
        pipeline, X, y, cv=cv, scoring='f1_macro', n_jobs=-1
    )

    # FIX 12: macro-F1 evaluation
    y_pred   = pipeline.predict(X_test)
    macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    print(f"\n  Holdout macro-F1 : {macro_f1:.4f}")
    print(f"  CV macro-F1      : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")

    # FIX 3: 1.0000 on demo data = leakage warning
    if macro_f1 == 1.0 and cv_f1.mean() == 1.0:
        print(f"\n  ⚠  PERFECT SCORE WARNING")
        print(f"     macro-F1 = 1.0000 on both holdout and CV.")
        print(f"     On real data this almost never happens.")
        print(f"     Likely cause: Sub_Type name appears verbatim in Case_Text.")
        print(f"     Verify your text does not contain the label you are predicting.")

    print(f"\n{classification_report(y_test, y_pred, zero_division=0)}")

    # FIX 11: confusion matrix
    labels = sorted(y.unique())
    cm     = confusion_matrix(y_test, y_pred, labels=labels)
    col_w  = max(len(l) for l in labels) + 2
    print(f"  Confusion matrix  (rows=actual, cols=predicted)")
    print(f"  {'':>{col_w}}", end='')
    for l in labels:
        print(f"  {l:>{col_w}}", end='')
    print()
    for i, row in enumerate(cm):
        print(f"  {labels[i]:>{col_w}}", end='')
        for val in row:
            print(f"  {val:>{col_w}}", end='')
        print()

    # Interpretation
    print(f"\n  Interpretation:")
    if macro_f1 >= 0.75:
        print(f"  ✓ macro-F1 {macro_f1:.2f} — strong.")
    elif macro_f1 >= 0.55:
        print(f"  ~ macro-F1 {macro_f1:.2f} — acceptable baseline.")
        print(f"    Try: more data for minority class, SMOTE, XGBoost.")
    else:
        print(f"  ✗ macro-F1 {macro_f1:.2f} — weak.")
        print(f"    Possible causes:")
        print(f"    1. Rare classes too few — target data collection first")
        print(f"    2. Case_Text has low discriminating signal")
        print(f"    3. Try Legal-BERT embeddings instead of TF-IDF")

    return {
        'pipeline':   pipeline,
        'X_train': X_train, 'X_test':  X_test,
        'y_train': y_train, 'y_test':  y_test,
        'y_pred':  y_pred,
        'macro_f1':   macro_f1,
        'cv_f1_mean': float(cv_f1.mean()),
        'cv_f1_std':  float(cv_f1.std()),
        'classes':    list(pipeline.classes_),
    }


# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 5 — HIERARCHICAL PREDICTION
# ═══════════════════════════════════════════════════════════════════════════════

def predict_one(
    row: pd.Series,
    l0_pipeline:       Pipeline,
    l1_pipeline:       Pipeline,
    l2_civil_pipeline: Pipeline,
    l2_crim_pipeline:  Pipeline,
    threshold: float = CONFIDENCE_THRESHOLD,
) -> Dict[str, object]:
    """
    Predict a single row through the full 3-level hierarchy.

    Flow:
        L0 predict (5-class)
            ↓ conf < threshold → OTHER at all levels
            ↓ CONTRACT / PROPERTY / TORTS → final, no L2
            ↓ CIVIL / CRIMINAL →
                L1 predict (binary)
                    ↓ conf < threshold → OTHER at L2
                    ↓ CIVIL   → L2a predict → APPELLATE / TRIAL_ENFORCEMENT
                    ↓ CRIMINAL → L2b predict → GENERAL_CRIMINAL / SPECIFIC_OFFENSE
                        ↓ L2 conf < threshold → OTHER

    FIX 4: confidence gating at every level
    FIX 5: L1→L2 hierarchy enforced — impossible combos blocked
    """
    X_row = pd.DataFrame([row[ALL_FEATURES]])

    # ── L0: 5-class ───────────────────────────────────────────────────────
    l0_proba  = l0_pipeline.predict_proba(X_row)[0]
    l0_labels = l0_pipeline.classes_
    l0_idx    = int(np.argmax(l0_proba))
    l0_pred   = l0_labels[l0_idx]
    l0_conf   = float(l0_proba[l0_idx])

    base = {
        'L0_Predicted':  l0_pred,
        'L0_Confidence': round(l0_conf, 4),
        'L1_Predicted':  None,
        'L1_Confidence': None,
        'L2_Predicted':  None,
        'L2_Confidence': None,
        'Low_Conf_Flag': False,
        'Hierarchy_Valid': True,
    }

    if l0_conf < threshold:
        base.update({'L0_Predicted': 'OTHER', 'Low_Conf_Flag': True})
        return base

    # Cases not routed to L2
    if l0_pred not in L2_HAS_SUBCLASSIFIER:
        return base

    # ── L1: CIVIL vs CRIMINAL ─────────────────────────────────────────────
    l1_proba  = l1_pipeline.predict_proba(X_row)[0]
    l1_labels = l1_pipeline.classes_
    l1_idx    = int(np.argmax(l1_proba))
    l1_pred   = l1_labels[l1_idx]
    l1_conf   = float(l1_proba[l1_idx])

    base['L1_Predicted']  = l1_pred
    base['L1_Confidence'] = round(l1_conf, 4)

    # FIX 5: L0 and L1 must agree — if they disagree, trust L0
    if l1_pred != l0_pred:
        base['Hierarchy_Valid'] = False
        l1_pred = l0_pred     # enforce L0 prediction

    if l1_conf < threshold:
        base['L2_Predicted'] = 'OTHER'
        base['Low_Conf_Flag'] = True
        return base

    # ── L2: route by L1 prediction ────────────────────────────────────────
    l2_pipeline = l2_civil_pipeline if l1_pred == 'CIVIL' else l2_crim_pipeline

    l2_proba  = l2_pipeline.predict_proba(X_row)[0]
    l2_labels = l2_pipeline.classes_
    l2_idx    = int(np.argmax(l2_proba))
    l2_pred   = l2_labels[l2_idx]
    l2_conf   = float(l2_proba[l2_idx])

    # FIX 5: L2 label must be a valid child of L1
    valid_l2 = L1_TO_L2_CLASSES.get(l1_pred, [])
    if l2_pred not in valid_l2:
        l2_pred = valid_l2[0] if valid_l2 else 'OTHER'
        base['Hierarchy_Valid'] = False

    if l2_conf < threshold:
        l2_pred = 'OTHER'
        base['Low_Conf_Flag'] = True

    base['L2_Predicted']  = l2_pred
    base['L2_Confidence'] = round(l2_conf, 4)
    return base


def predict_batch(
    df_new: pd.DataFrame,
    l0_pipeline:       Pipeline,
    l1_pipeline:       Pipeline,
    l2_civil_pipeline: Pipeline,
    l2_crim_pipeline:  Pipeline,
    threshold: float = CONFIDENCE_THRESHOLD,
) -> pd.DataFrame:
    """
    Predict all rows.  Returns df_new with prediction columns appended.
    FIX 10: returns confidence scores + hierarchy validity flag for every row.
    """
    results = [
        predict_one(
            row, l0_pipeline, l1_pipeline,
            l2_civil_pipeline, l2_crim_pipeline, threshold
        )
        for _, row in df_new.iterrows()
    ]
    pred_df = pd.DataFrame(results)
    return pd.concat([df_new.reset_index(drop=True), pred_df], axis=1)


# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 6 — FULL PIPELINE ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════════════

def run_pipeline(
    df: pd.DataFrame,
    sub_type_col: str = 'Sub_Type',
    test_size: float = TEST_SIZE,
) -> dict:
    """
    End-to-end entry point.

    Required columns: Sub_Type, Case_Text, Court, Year

    Usage with real data:
        df      = pd.read_csv('your_cases.csv')
        results = run_pipeline(df)

    Trained pipelines:
        results['l0_pipeline']       — 5-class L0 (Case_Type)
        results['l1_pipeline']       — binary L1 (CIVIL vs CRIMINAL)
        results['l2_civil_pipeline'] — binary L2a (Civil sub-types)
        results['l2_crim_pipeline']  — binary L2b (Criminal sub-types)

    Predict new cases:
        preds = predict_batch(df_new, **{k: results[k] for k in
                    ['l0_pipeline','l1_pipeline',
                     'l2_civil_pipeline','l2_crim_pipeline']})
    """

    print("STEP 1/5  Applying L0 / L1 / L2 mappings...")
    df_mapped = apply_mappings(df, sub_type_col=sub_type_col)

    # ── Step 2: L0 — 5-class Case_Type classifier ─────────────────────────
    print("\nSTEP 2/5  Training L0 — 5-class case type (CIVIL / CRIMINAL / "
          "CONTRACT / PROPERTY / TORTS)...")
    df_l0 = df_mapped[df_mapped['L0_Label'] != 'OTHER'].copy()
    l0_res = train_level(df_l0, 'L0_Label', 'L0: 5-class case type', test_size)

    # ── Step 3: L1 — CIVIL vs CRIMINAL ────────────────────────────────────
    print("\nSTEP 3/5  Training L1 — CIVIL vs CRIMINAL...")
    df_l1 = df_mapped[df_mapped['L1_Label'].notna()].copy()
    l1_res = train_level(df_l1, 'L1_Label', 'L1: CIVIL vs CRIMINAL', test_size)

    # ── Step 4: L2 Civil ──────────────────────────────────────────────────
    print("\nSTEP 4/5  Training L2a — APPELLATE vs TRIAL_ENFORCEMENT (Civil only)...")
    df_civil = df_mapped[df_mapped['L0_Label'] == 'CIVIL'].copy()
    l2_civil_res = train_level(
        df_civil, 'L2_Civil', 'L2a: Civil sub-types', test_size
    )

    # ── Step 5: L2 Criminal ───────────────────────────────────────────────
    print("\nSTEP 5/5  Training L2b — GENERAL_CRIMINAL vs SPECIFIC_OFFENSE "
          "(Criminal only)...")
    df_crim = df_mapped[df_mapped['L0_Label'] == 'CRIMINAL'].copy()
    l2_crim_res = train_level(
        df_crim, 'L2_Criminal', 'L2b: Criminal sub-types', test_size
    )

    # ── Summary ───────────────────────────────────────────────────────────
    print(f"\n{'═'*60}")
    print(f"  PIPELINE COMPLETE — SUMMARY")
    print(f"{'═'*60}")
    print(f"  {'Level':<38} {'Holdout F1':>10}  {'CV F1 ± std':>16}")
    print(f"  {'─'*68}")
    for name, res in [
        ('L0: 5-class case type',              l0_res),
        ('L1: CIVIL vs CRIMINAL',              l1_res),
        ('L2a: APPELLATE vs TRIAL_ENFORCEMENT',l2_civil_res),
        ('L2b: GENERAL vs SPECIFIC_OFFENSE',   l2_crim_res),
    ]:
        cv_str = f"{res['cv_f1_mean']:.4f} ± {res['cv_f1_std']:.4f}"
        print(f"  {name:<38} {res['macro_f1']:>10.4f}  {cv_str:>16}")

    print(f"\n  Confidence threshold : {CONFIDENCE_THRESHOLD}")
    print(f"  → Predictions below this → 'OTHER' at that level")
    print(f"  → Override: results['threshold'] = 0.40  (lower = fewer OTHER)")

    return {
        'df_mapped':          df_mapped,
        'l0':                 l0_res,
        'l1':                 l1_res,
        'l2_civil':           l2_civil_res,
        'l2_criminal':        l2_crim_res,
        'l0_pipeline':        l0_res['pipeline'],
        'l1_pipeline':        l1_res['pipeline'],
        'l2_civil_pipeline':  l2_civil_res['pipeline'],
        'l2_crim_pipeline':   l2_crim_res['pipeline'],
        'threshold':          CONFIDENCE_THRESHOLD,
    }


# ═══════════════════════════════════════════════════════════════════════════════
# DEMO  — leakage-free synthetic data
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':

    # ── FIX 3: demo text does NOT embed Sub_Type name ──────────────────────
    # Each Sub_Type has a distinct vocabulary that doesn't repeat its own name.
    # On real data Case_Text will be the full case document.

    SUBTYPE_VOCAB: Dict[str, List[str]] = {
        # CIVIL — procedural language
        'Appeal': [
            "plaintiff filed notice seeking review of lower court judgment",
            "error committed below requires reversal on points of law",
            "transcript certified and docketed for appellate examination",
            "counsel assigned errors in the ruling below",
        ],
        'Demurrer': [
            "defendant demurred to the declaration as legally insufficient",
            "pleading on its face discloses no cause of action in law",
            "court sustained the objection to the form of the writ",
            "matter of law raised on the face of the pleading",
        ],
        'Fieri Facias': [
            "writ directed sheriff to levy on goods and chattels of debtor",
            "execution issued on the judgment against personal property",
            "sheriff returned nulla bona on the writ of execution",
        ],
        'General Execution': [
            "writ of execution issued on judgment of the court",
            "levy made upon lands and tenements of the judgment debtor",
            "execution returned unsatisfied in part by the officer",
        ],
        'General Civil Procedure': [
            "court entered order governing the course of proceedings",
            "motion filed and argued respecting the form of proceeding",
            "procedural rule applied to the conduct of the suit",
        ],
        'Pleading': [
            "plaintiff filed declaration setting forth the cause of action",
            "special plea entered traversing the material allegations",
            "replication filed to the plea in bar of the action",
        ],
        'Capias': [
            "capias issued commanding the sheriff to arrest the body",
            "defendant taken into custody on the original process",
            "writ of capias ad respondendum served on the defendant",
        ],
        'Writ': [
            "writ issued out of chancery commanding attendance",
            "original writ tested and returned by the officer",
            "writ of mandamus directed to the inferior tribunal",
        ],
        'Attachment': [
            "attachment issued against the property of the absent debtor",
            "garnishment served on the trustee holding the funds",
            "property attached to secure the debt pending suit",
        ],
        'Scire Facias': [
            "scire facias to revive the dormant judgment of record",
            "writ commanding the party to show cause against revival",
            "judgment revived on scire facias returned duly served",
        ],
        # CRIMINAL — offense-specific language
        'General Criminal': [
            "court examined the record and proceedings below",
            "bill of indictment found by the grand jury at term",
            "plea of not guilty entered and jury impaneled",
            "verdict returned by the jury on the issue joined",
        ],
        'Indictment': [
            "grand jury returned true bill charging the offense",
            "indictment alleged the act with requisite certainty",
            "demurrer to the indictment overruled by the court",
            "bill quashed for variance between allegation and proof",
        ],
        'Homicide': [
            "deceased received mortal wound from the hand of defendant",
            "coroner's jury found death by violent means",
            "malice aforethought charged in the first count of the bill",
            "witness testified to the blow inflicted on the deceased",
        ],
        'Assault & Battery': [
            "defendant struck the complainant with force and arms",
            "blow inflicted causing bodily harm to the prosecutor",
            "affray in the public street resulting in injury",
        ],
        'Larceny': [
            "goods taken feloniously from the possession of the owner",
            "indictment charged taking with intent to permanently deprive",
            "property stolen from the store of the merchant",
        ],
        'Perjury': [
            "defendant swore falsely in a judicial proceeding",
            "oath administered and false statement knowingly made",
            "materiality of the false testimony established at trial",
        ],
        'Riot': [
            "three or more persons assembled with unlawful force",
            "riotous assembly dispersed by the magistrate",
            "participants acted in concert terrorizing the neighborhood",
        ],
        'Counterfeiting': [
            "spurious notes passed as genuine currency of the bank",
            "forged instrument uttered with knowledge of the falsity",
            "die and press discovered in the possession of defendant",
        ],
        'Burglary/Robbery': [
            "dwelling house broken and entered in the night season",
            "victim robbed by force and putting in fear on the highway",
            "taking accomplished by violence and intimidation",
        ],
        'Affray': [
            "mutual combat in a public place disturbing the peace",
            "parties engaged in fighting to the terror of the citizens",
            "brawl occurred in a place of public resort",
        ],
        # CONTRACT — commercial language
        'Promissory Note': [
            "maker executed note payable at sixty days to the order of payee",
            "endorsement transferred the instrument to the holder",
            "note dishonored and protest made at maturity",
        ],
        'Bond': [
            "obligor bound in the penal sum with condition annexed",
            "bond executed with sufficient surety approved by the court",
            "breach of condition forfeited the penalty of the bond",
        ],
        'Debt Collection': [
            "account stated between the parties showing balance due",
            "indebtedness acknowledged and payment refused",
            "assumpsit brought to recover the sum certain owed",
        ],
        'General Contract': [
            "agreement entered into for valuable consideration",
            "breach alleged for non-performance of the covenant",
            "mutual promises formed the basis of the contract",
        ],
        'Usury': [
            "interest charged in excess of the legal rate allowed",
            "usurious contract void under the statute",
            "lender exacted illegal premium on the principal sum",
        ],
        # PROPERTY — real property language
        'Ejectment': [
            "plaintiff claimed title and right of possession of the land",
            "ouster alleged and entry demanded before suit brought",
            "lessor of the plaintiff recovered on the better title",
        ],
        'Title Dispute': [
            "competing grants from the commonwealth produced the controversy",
            "survey and patent examined to determine priority of title",
            "elder grant held superior by reason of prior location",
        ],
        'Mortgage Foreclosure': [
            "mortgagor defaulted on the condition of the deed",
            "bill filed to foreclose the equity of redemption",
            "sale ordered by the court to satisfy the mortgage debt",
        ],
        'General Property': [
            "ownership of the tract disputed between adjoining claimants",
            "deed of conveyance recorded and its validity contested",
            "boundary line established by survey and ancient marks",
        ],
        # TORTS — personal wrong language
        'Slander': [
            "words spoken imputed a crime to the plaintiff",
            "defamatory words uttered in the hearing of witnesses",
            "special damages proved from the slanderous words",
        ],
        'Defamation': [
            "false statement injured the reputation and credit of plaintiff",
            "publication made to third parties without lawful justification",
            "imputation of dishonesty caused loss of business",
        ],
        'Libel': [
            "written matter published charging plaintiff with misconduct",
            "printed words held actionable per se by the court",
            "newspaper article damaged the standing of the plaintiff",
        ],
        'Trespass': [
            "defendant entered upon the close of the plaintiff without leave",
            "wrongful entry caused damage to the growing crops",
            "trespass vi et armis alleged with force and arms",
        ],
        'General Torts': [
            "wrong done to the person or property of the plaintiff",
            "duty of care breached causing injury compensable in damages",
        ],
        'Negligence': [
            "duty neglected and injury proximately resulted therefrom",
            "failure to exercise ordinary care caused the harm",
            "defendant's carelessness created an unreasonable risk",
        ],
        'Conversion': [
            "personal property unlawfully taken and converted to own use",
            "owner dispossessed without authority or consent",
            "trover brought for the value of the goods converted",
        ],
    }

    real_counts = {
        'Appeal': 1405, 'General Criminal': 816, 'Demurrer': 555,
        'Promissory Note': 254, 'Bond': 105, 'Ejectment': 96,
        'Slander': 93, 'Title Dispute': 91, 'Larceny': 67,
        'Homicide': 66, 'Mortgage Foreclosure': 66, 'Assault & Battery': 53,
        'Defamation': 49, 'Indictment': 48, 'General Property': 37,
        'Fieri Facias': 26, 'Debt Collection': 23, 'General Contract': 22,
        'Perjury': 20, 'General Execution': 18, 'Trespass': 16,
        'Libel': 13, 'Riot': 12, 'Counterfeiting': 12, 'General Torts': 12,
        'Usury': 9, 'General Civil Procedure': 8, 'Negligence': 8,
        'Pleading': 6, 'Burglary/Robbery': 6, 'Conversion': 6,
        'Capias': 3, 'Writ': 2, 'Affray': 1, 'Attachment': 1, 'Scire Facias': 1,
    }

    rng  = np.random.default_rng(seed=42)
    rows = []
    for sub_type, count in real_counts.items():
        vocab = SUBTYPE_VOCAB.get(sub_type, [f"matter came before the court case {sub_type}"])
        for i in range(count):
            # FIX 3: rotate through distinct vocabulary sentences
            # Sub_Type name is NOT embedded in the text
            n_sent = min(4, len(vocab))
            chosen = rng.choice(vocab, size=n_sent, replace=True)
            text   = ' '.join(chosen) + f' Index {i}.'
            rows.append({
                'Sub_Type':  sub_type,
                'Case_Text': text,
                'Court':     'Supreme Court' if i % 3 != 0 else 'Circuit Court',
                'Year':      1820 + (i % 60),
            })

    df_demo = pd.DataFrame(rows)
    print(f"Demo dataset : {len(df_demo):,} rows, "
          f"{df_demo['Sub_Type'].nunique()} unique Sub_Types")
    print(f"Note: demo text does NOT embed Sub_Type name — leakage-free.\n")

    results = run_pipeline(df_demo)

    # ── Sample predictions ─────────────────────────────────────────────────
    print(f"\n{'═'*60}")
    print(f"  SAMPLE PREDICTIONS  (6 rows, 2 per domain)")
    print(f"{'═'*60}")

    sample_types = ['Appeal', 'Demurrer', 'Homicide', 'Larceny',
                    'Promissory Note', 'Slander']
    sample = df_demo[df_demo['Sub_Type'].isin(sample_types)].groupby(
        'Sub_Type', group_keys=False
    ).apply(lambda g: g.sample(1, random_state=1)).reset_index(drop=True)

    pred_df = predict_batch(
        sample,
        results['l0_pipeline'],
        results['l1_pipeline'],
        results['l2_civil_pipeline'],
        results['l2_crim_pipeline'],
    )

    for _, row in pred_df.iterrows():
        flag = '⚠ LOW CONF' if row['Low_Conf_Flag'] else '✓'
        hier = '✓' if row['Hierarchy_Valid'] else '⚠ HIERARCHY MISMATCH'
        print(f"\n  Sub_Type      : {row['Sub_Type']}")
        print(f"  L0 Predicted  : {row['L0_Predicted']:<20} "
              f"conf={row['L0_Confidence']:.3f}")
        print(f"  L1 Predicted  : {str(row['L1_Predicted']):<20} "
              f"conf={row.get('L1_Confidence')}")
        print(f"  L2 Predicted  : {str(row['L2_Predicted']):<20} "
              f"conf={row.get('L2_Confidence')}")
        print(f"  Status        : {flag}  |  Hierarchy: {hier}")

Demo dataset : 4,026 rows, 36 unique Sub_Types
Note: demo text does NOT embed Sub_Type name — leakage-free.

STEP 1/5  Applying L0 / L1 / L2 mappings...

════════════════════════════════════════════════════════════
  MAPPING REPORT  (n=4,026)
════════════════════════════════════════════════════════════

  CIVIL          2025 rows  ( 50.3%)  ████████████████
    ├── APPELLATE               1408
    ├── TRIAL_ENFORCEMENT        617

  CRIMINAL       1101 rows  ( 27.3%)  █████████
    ├── GENERAL_CRIMINAL         864
    ├── SPECIFIC_OFFENSE         237

  CONTRACT        413 rows  ( 10.3%)  ███

  PROPERTY        290 rows  (  7.2%)  ██

  TORTS           197 rows  (  4.9%)  █

  OTHER             0 rows  (  0.0%)  █

  IMBALANCE CHECK
  CIVIL       imbalance    2.3x  [OK]
  CRIMINAL    imbalance    3.6x  [OK]

STEP 2/5  Training L0 — 5-class case type (CIVIL / CRIMINAL / CONTRACT / PROPERTY / TORTS)...

════════════════════════════════════════════════════════════
  L0: 5-class case type 

In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 5

# =========================
# LABEL MAPPINGS (from Sub_Type)
# =========================

L0_MAPPING = {
    # CIVIL
    'Appeal':'CIVIL','Demurrer':'CIVIL','Writ':'CIVIL','Pleading':'CIVIL',
    'Attachment':'CIVIL','Scire Facias':'CIVIL',

    # CRIMINAL
    'Homicide':'CRIMINAL','Larceny':'CRIMINAL','Perjury':'CRIMINAL',
    'Riot':'CRIMINAL','Assault & Battery':'CRIMINAL','Indictment':'CRIMINAL',

    # CONTRACT
    'Promissory Note':'CONTRACT','Bond':'CONTRACT','Debt Collection':'CONTRACT',

    # PROPERTY
    'Ejectment':'PROPERTY','Title Dispute':'PROPERTY',

    # TORTS
    'Slander':'TORTS','Negligence':'TORTS','Trespass':'TORTS'
}

L1_MAPPING = {k:v for k,v in L0_MAPPING.items() if v in ['CIVIL','CRIMINAL']}

In [12]:
def prepare_data(df: pd.DataFrame):

    df = df.copy()

    # --- ONLY FEATURE USED ---
    X = df["Case_Text"]

    # --- LABELS ---
    df["L0"] = df["Sub_Type"].map(L0_MAPPING)
    df["L1"] = df["Sub_Type"].map(L1_MAPPING)

    return df, X

In [13]:
def build_model():

    return Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=20000,
            ngram_range=(1,2),
            min_df=2
        )),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        ))
    ])


def train_level(X, y, name="LEVEL"):

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE
    )

    model = build_model()
    model.fit(X_train, y_train)

    # CV
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring="f1_macro")

    # TEST
    preds = model.predict(X_test)

    print(f"\n════════ {name} ════════")
    print("CV F1:", cv_scores.mean(), "±", cv_scores.std())
    print("TEST F1:", f1_score(y_test, preds, average="macro"))
    print(classification_report(y_test, preds))

    return model

In [14]:
# df must already exist (REAL DATA ONLY)

df, X = prepare_data(df)

# -------------------------
# L0 MODEL (ALL CLASSES)
# -------------------------
df_l0 = df.dropna(subset=["L0"])
model_l0 = train_level(df_l0["Case_Text"], df_l0["L0"], "L0 - CASE TYPE")

# -------------------------
# L1 MODEL (CIVIL vs CRIMINAL ONLY)
# -------------------------
df_l1 = df.dropna(subset=["L1"])
model_l1 = train_level(df_l1["Case_Text"], df_l1["L1"], "L1 - CIVIL vs CRIMINAL")


════════ L0 - CASE TYPE ════════
CV F1: 0.754333867194051 ± 0.0048463926986665965
TEST F1: 0.7613601845880641
              precision    recall  f1-score   support

       CIVIL       0.90      0.75      0.82      1061
    CONTRACT       0.62      0.71      0.66       399
    CRIMINAL       0.75      0.88      0.81       164
    PROPERTY       0.64      0.74      0.69       252
       TORTS       0.77      0.91      0.83       295

    accuracy                           0.77      2171
   macro avg       0.74      0.80      0.76      2171
weighted avg       0.79      0.77      0.77      2171


════════ L1 - CIVIL vs CRIMINAL ════════
CV F1: 0.9107844501335485 ± 0.017424797152091893
TEST F1: 0.9094212389261656
              precision    recall  f1-score   support

       CIVIL       0.99      0.96      0.97      1062
    CRIMINAL       0.79      0.91      0.85       164

    accuracy                           0.96      1226
   macro avg       0.89      0.94      0.91      1226
weighted 

In [15]:
# ╔══════════════════════════════════════════════════════════════╗
# ║ CELL 1 — IMPORTS + CONFIGURATION + LABEL MAPPINGS          ║
# ╚══════════════════════════════════════════════════════════════╝

import pandas as pd
import numpy as np
import warnings

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================

TEXT_FEATURE = "Case_Text"
CAT_FEATURES = ["Court"]
NUM_FEATURES = ["Year"]

ALL_FEATURES = [TEXT_FEATURE] + CAT_FEATURES + NUM_FEATURES

CONFIDENCE_THRESHOLD = 0.45
TEST_SIZE = 0.20
CV_FOLDS = 5
RANDOM_STATE = 42
MIN_CLASS_ROWS = 30


# ============================================================
# LEAKAGE GUARD
# ============================================================

LEAKAGE_COLUMNS = [
    "Sub_Type",
    "L0_Label",
    "L1_Label",
    "L2_Civil",
    "L2_Criminal",
]


# ============================================================
# HIERARCHICAL MAPPINGS
# ============================================================

# ---------------------------
# LEVEL 0
# ---------------------------

L0_MAPPING = {

    # CIVIL
    'Appeal':'CIVIL',
    'Demurrer':'CIVIL',
    'Fieri Facias':'CIVIL',
    'General Execution':'CIVIL',
    'General Civil Procedure':'CIVIL',
    'Pleading':'CIVIL',
    'Capias':'CIVIL',
    'Writ':'CIVIL',
    'Attachment':'CIVIL',
    'Scire Facias':'CIVIL',

    # CRIMINAL
    'General Criminal':'CRIMINAL',
    'Indictment':'CRIMINAL',
    'Homicide':'CRIMINAL',
    'Assault & Battery':'CRIMINAL',
    'Larceny':'CRIMINAL',
    'Perjury':'CRIMINAL',
    'Riot':'CRIMINAL',
    'Counterfeiting':'CRIMINAL',
    'Burglary/Robbery':'CRIMINAL',
    'Affray':'CRIMINAL',

    # CONTRACT
    'Promissory Note':'CONTRACT',
    'Bond':'CONTRACT',
    'Debt Collection':'CONTRACT',
    'General Contract':'CONTRACT',
    'Usury':'CONTRACT',

    # PROPERTY
    'Ejectment':'PROPERTY',
    'Title Dispute':'PROPERTY',
    'Mortgage Foreclosure':'PROPERTY',
    'General Property':'PROPERTY',

    # TORTS
    'Slander':'TORTS',
    'Defamation':'TORTS',
    'Libel':'TORTS',
    'Trespass':'TORTS',
    'General Torts':'TORTS',
    'Negligence':'TORTS',
    'Conversion':'TORTS',
}


# ---------------------------
# LEVEL 1
# ---------------------------

L1_MAPPING = {
    k:v for k,v in L0_MAPPING.items()
    if v in ["CIVIL", "CRIMINAL"]
}


# ---------------------------
# LEVEL 2 CIVIL
# ---------------------------

L2_CIVIL_MAPPING = {

    'Appeal':'APPELLATE',
    'Writ':'APPELLATE',
    'Scire Facias':'APPELLATE',

    'Demurrer':'TRIAL_ENFORCEMENT',
    'Fieri Facias':'TRIAL_ENFORCEMENT',
    'General Execution':'TRIAL_ENFORCEMENT',
    'General Civil Procedure':'TRIAL_ENFORCEMENT',
    'Pleading':'TRIAL_ENFORCEMENT',
    'Capias':'TRIAL_ENFORCEMENT',
    'Attachment':'TRIAL_ENFORCEMENT',
}


# ---------------------------
# LEVEL 2 CRIMINAL
# ---------------------------

L2_CRIMINAL_MAPPING = {

    'General Criminal':'GENERAL_CRIMINAL',
    'Indictment':'GENERAL_CRIMINAL',

    'Homicide':'SPECIFIC_OFFENSE',
    'Assault & Battery':'SPECIFIC_OFFENSE',
    'Larceny':'SPECIFIC_OFFENSE',
    'Perjury':'SPECIFIC_OFFENSE',
    'Riot':'SPECIFIC_OFFENSE',
    'Counterfeiting':'SPECIFIC_OFFENSE',
    'Burglary/Robbery':'SPECIFIC_OFFENSE',
    'Affray':'SPECIFIC_OFFENSE',
}


# ============================================================
# APPLY LABELS TO REAL DATAFRAME df
# ============================================================

df = df.copy()

df["L0_Label"] = df["Sub_Type"].map(L0_MAPPING).fillna("OTHER")

df["L1_Label"] = df["Sub_Type"].map(L1_MAPPING)

df["L2_Civil"] = df["Sub_Type"].map(L2_CIVIL_MAPPING)

df["L2_Criminal"] = df["Sub_Type"].map(L2_CRIMINAL_MAPPING)

print("DATA READY")
print(df.head())

DATA READY
   Case_ID Case_Name Source_Folder  Year                       Court  \
0      347   0347-01       fil3212  1959  Connecticut Superior Court   
1      477   0477-01       fil3212  1958  Connecticut Superior Court   
2      217   0217-01       fil3212  1959  Connecticut Superior Court   
3      132   0132-01       fil3212  1958  Connecticut Superior Court   
4      487   0487-01       fil3212  1960  Connecticut Superior Court   

                                           Case_Text  Case_Text_Full_Length  \
0  James Yaworski et al. v. Town of Canterbury et...                   8903   
1  State of Connecticut v. John Salta Decided Oct...                   2851   
2  The Bacon Memorial Home v. John J. Bracken, At...                   8245   
3  Hazel M. Harris v. Housing Authority of the Ci...                   2903   
4  Yolande Berube v. The Salvation Army, Inc. Sup...                   1483   

                       Verdict        Case_Type           Sub_Type  ...  \
0     

In [16]:
# ╔══════════════════════════════════════════════════════════════╗
# ║ CELL 2 — PIPELINE + TRAINING FUNCTION                       ║
# ╚══════════════════════════════════════════════════════════════╝

# ============================================================
# FEATURE SAFETY
# ============================================================

def safe_X(dataframe):

    X = dataframe[ALL_FEATURES].copy()

    leaked = [c for c in LEAKAGE_COLUMNS if c in X.columns]

    if leaked:
        raise ValueError(f"LEAKAGE DETECTED: {leaked}")

    return X


# ============================================================
# BUILD MODEL PIPELINE
# ============================================================

def build_pipeline():

    preprocessor = ColumnTransformer(

        transformers=[

            (
                "tfidf",

                TfidfVectorizer(
                    max_features=10000,
                    ngram_range=(1,2),
                    min_df=2,
                    sublinear_tf=True,
                    strip_accents="unicode",
                ),

                TEXT_FEATURE
            ),

            (
                "cat",

                OneHotEncoder(
                    handle_unknown="ignore"
                ),

                CAT_FEATURES
            ),

            (
                "num",
                "passthrough",
                NUM_FEATURES
            )
        ]
    )

    pipeline = Pipeline([

        ("preprocessor", preprocessor),

        ("classifier",

            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                solver="lbfgs",
                multi_class="multinomial",
                random_state=RANDOM_STATE
            )
        )
    ])

    return pipeline


# ============================================================
# TRAINING FUNCTION
# ============================================================

def train_level(df_subset, label_col, level_name):

    print("\n" + "="*60)
    print(level_name)
    print("="*60)

    df_subset = df_subset.dropna(subset=[label_col]).copy()

    X = safe_X(df_subset)

    y = df_subset[label_col]

    print("\nCLASS DISTRIBUTION")
    print(y.value_counts())

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(

        X,
        y,

        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE
    )

    pipeline = build_pipeline()

    pipeline.fit(X_train, y_train)

    # predictions
    y_pred = pipeline.predict(X_test)

    # macro f1
    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    # CV
    cv = StratifiedKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    cv_scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1
    )

    print(f"\nHoldout Macro-F1 : {macro_f1:.4f}")

    print(
        f"CV Macro-F1      : "
        f"{cv_scores.mean():.4f} ± {cv_scores.std():.4f}"
    )

    print("\nCLASSIFICATION REPORT\n")

    print(classification_report(y_test, y_pred))

    print("\nCONFUSION MATRIX\n")

    labels = sorted(y.unique())

    cm = confusion_matrix(y_test, y_pred, labels=labels)

    cm_df = pd.DataFrame(
        cm,
        index=labels,
        columns=labels
    )

    print(cm_df)

    return {
        "pipeline":pipeline,
        "macro_f1":macro_f1,
        "cv_mean":cv_scores.mean(),
        "cv_std":cv_scores.std(),
        "classes":pipeline.classes_
    }

In [17]:
# ╔══════════════════════════════════════════════════════════════╗
# ║ CELL 3 — TRAIN FULL HIERARCHICAL SYSTEM                     ║
# ╚══════════════════════════════════════════════════════════════╝

# ============================================================
# L0 — 5 CLASS
# ============================================================

df_l0 = df[df["L0_Label"] != "OTHER"].copy()

l0_results = train_level(
    df_l0,
    "L0_Label",
    "L0 — CASE TYPE CLASSIFIER"
)


# ============================================================
# L1 — CIVIL vs CRIMINAL
# ============================================================

df_l1 = df[df["L1_Label"].notna()].copy()

l1_results = train_level(
    df_l1,
    "L1_Label",
    "L1 — CIVIL vs CRIMINAL"
)


# ============================================================
# L2a — CIVIL SUBTYPES
# ============================================================

df_civil = df[df["L0_Label"] == "CIVIL"].copy()

l2_civil_results = train_level(
    df_civil,
    "L2_Civil",
    "L2a — CIVIL SUBCLASSIFIER"
)


# ============================================================
# L2b — CRIMINAL SUBTYPES
# ============================================================

df_criminal = df[df["L0_Label"] == "CRIMINAL"].copy()

l2_criminal_results = train_level(
    df_criminal,
    "L2_Criminal",
    "L2b — CRIMINAL SUBCLASSIFIER"
)


# ============================================================
# SAVE PIPELINES
# ============================================================

l0_pipeline = l0_results["pipeline"]

l1_pipeline = l1_results["pipeline"]

l2_civil_pipeline = l2_civil_results["pipeline"]

l2_criminal_pipeline = l2_criminal_results["pipeline"]

print("\nTRAINING COMPLETE")


L0 — CASE TYPE CLASSIFIER

CLASS DISTRIBUTION
L0_Label
CIVIL       5542
CRIMINAL    4160
CONTRACT    2402
TORTS       2055
PROPERTY    1880
Name: count, dtype: int64

Holdout Macro-F1 : 0.6465
CV Macro-F1      : 0.6520 ± 0.0168

CLASSIFICATION REPORT

              precision    recall  f1-score   support

       CIVIL       0.75      0.64      0.69      1109
    CONTRACT       0.51      0.65      0.57       480
    CRIMINAL       0.79      0.58      0.67       832
    PROPERTY       0.63      0.72      0.67       376
       TORTS       0.53      0.79      0.63       411

    accuracy                           0.65      3208
   macro avg       0.64      0.68      0.65      3208
weighted avg       0.68      0.65      0.66      3208


CONFUSION MATRIX

          CIVIL  CONTRACT  CRIMINAL  PROPERTY  TORTS
CIVIL       707       216        63        80     43
CONTRACT     52       311        40        60     17
CRIMINAL     88        41       483        12    208
PROPERTY     35        37  

In [18]:
# ╔══════════════════════════════════════════════════════════════╗
# ║ CELL 4 — HIERARCHICAL PREDICTION ON REAL DATA                ║
# ╚══════════════════════════════════════════════════════════════╝

# ============================================================
# HIERARCHICAL PREDICTION
# ============================================================

def hierarchical_predict(row):

    X_row = pd.DataFrame([row[ALL_FEATURES]])

    # ========================================================
    # LEVEL 0
    # ========================================================

    l0_probs = l0_pipeline.predict_proba(X_row)[0]

    l0_idx = np.argmax(l0_probs)

    l0_pred = l0_pipeline.classes_[l0_idx]

    l0_conf = l0_probs[l0_idx]

    result = {

        "L0_Predicted":l0_pred,
        "L0_Confidence":round(float(l0_conf),4),

        "L1_Predicted":None,
        "L1_Confidence":None,

        "L2_Predicted":None,
        "L2_Confidence":None,
    }

    # low confidence
    if l0_conf < CONFIDENCE_THRESHOLD:

        result["L0_Predicted"] = "OTHER"

        return result

    # ========================================================
    # NO LEVEL 2
    # ========================================================

    if l0_pred in ["CONTRACT", "PROPERTY", "TORTS"]:

        return result

    # ========================================================
    # LEVEL 1
    # ========================================================

    l1_probs = l1_pipeline.predict_proba(X_row)[0]

    l1_idx = np.argmax(l1_probs)

    l1_pred = l1_pipeline.classes_[l1_idx]

    l1_conf = l1_probs[l1_idx]

    result["L1_Predicted"] = l1_pred

    result["L1_Confidence"] = round(float(l1_conf),4)

    # ========================================================
    # LEVEL 2 ROUTING
    # ========================================================

    if l1_pred == "CIVIL":

        probs = l2_civil_pipeline.predict_proba(X_row)[0]

        idx = np.argmax(probs)

        pred = l2_civil_pipeline.classes_[idx]

        conf = probs[idx]

    else:

        probs = l2_criminal_pipeline.predict_proba(X_row)[0]

        idx = np.argmax(probs)

        pred = l2_criminal_pipeline.classes_[idx]

        conf = probs[idx]

    result["L2_Predicted"] = pred

    result["L2_Confidence"] = round(float(conf),4)

    return result


# ============================================================
# PREDICT SAMPLE ROWS
# ============================================================

sample_df = df.sample(10, random_state=42)

predictions = []

for _, row in sample_df.iterrows():

    pred = hierarchical_predict(row)

    predictions.append({

        "True_Sub_Type":row["Sub_Type"],

        "L0_Predicted":pred["L0_Predicted"],
        "L1_Predicted":pred["L1_Predicted"],
        "L2_Predicted":pred["L2_Predicted"],

        "L0_Confidence":pred["L0_Confidence"],
        "L1_Confidence":pred["L1_Confidence"],
        "L2_Confidence":pred["L2_Confidence"],
    })


pred_df = pd.DataFrame(predictions)

print(pred_df)


# ============================================================
# PREDICT NEW UNSEEN CASE
# ============================================================

new_case = pd.DataFrame([{

    "Case_Text":
        "defendant unlawfully took personal property "
        "with intent to permanently deprive the owner",

    "Court":"Supreme Court",

    "Year":1855
}])


new_prediction = hierarchical_predict(new_case.iloc[0])

print("\nNEW CASE PREDICTION\n")

for k,v in new_prediction.items():
    print(f"{k}: {v}")

             True_Sub_Type L0_Predicted L1_Predicted       L2_Predicted  \
0            General Torts        OTHER         None               None   
1                 Demurrer        CIVIL        CIVIL  TRIAL_ENFORCEMENT   
2         General Criminal     CRIMINAL     CRIMINAL   GENERAL_CRIMINAL   
3               Negligence        TORTS         None               None   
4                Ejectment     CONTRACT         None               None   
5         General Criminal        OTHER         None               None   
6  General Civil Procedure        OTHER         None               None   
7         General Criminal        TORTS         None               None   
8                     Bond     PROPERTY         None               None   
9         General Criminal     CRIMINAL     CRIMINAL   GENERAL_CRIMINAL   

   L0_Confidence  L1_Confidence  L2_Confidence  
0         0.4253            NaN            NaN  
1         0.5483         0.8983         0.9774  
2         0.6336         0.

In [19]:
# ╔══════════════════════════════════════════════════════════════╗
# ║ CELL 1 — IMPORTS + CONFIG + HIERARCHY LABELS               ║
# ╚══════════════════════════════════════════════════════════════╝

import pandas as pd
import numpy as np
import warnings

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)

warnings.filterwarnings("ignore")


# ============================================================
# CONFIGURATION
# ============================================================

TEXT_FEATURE = "Case_Text"

CONFIDENCE_THRESHOLD = 0.35
TEST_SIZE = 0.20
CV_FOLDS = 5
RANDOM_STATE = 42


# ============================================================
# LABEL MAPPINGS
# ============================================================

# ---------------------------
# LEVEL 0
# ---------------------------

L0_MAPPING = {

    # CIVIL
    'Appeal':'CIVIL',
    'Demurrer':'CIVIL',
    'Fieri Facias':'CIVIL',
    'General Execution':'CIVIL',
    'General Civil Procedure':'CIVIL',
    'Pleading':'CIVIL',
    'Capias':'CIVIL',
    'Writ':'CIVIL',
    'Attachment':'CIVIL',
    'Scire Facias':'CIVIL',

    # CRIMINAL
    'General Criminal':'CRIMINAL',
    'Indictment':'CRIMINAL',
    'Homicide':'CRIMINAL',
    'Assault & Battery':'CRIMINAL',
    'Larceny':'CRIMINAL',
    'Perjury':'CRIMINAL',
    'Riot':'CRIMINAL',
    'Counterfeiting':'CRIMINAL',
    'Burglary/Robbery':'CRIMINAL',
    'Affray':'CRIMINAL',

    # CONTRACT
    'Promissory Note':'CONTRACT',
    'Bond':'CONTRACT',
    'Debt Collection':'CONTRACT',
    'General Contract':'CONTRACT',
    'Usury':'CONTRACT',

    # PROPERTY
    'Ejectment':'PROPERTY',
    'Title Dispute':'PROPERTY',
    'Mortgage Foreclosure':'PROPERTY',
    'General Property':'PROPERTY',

    # TORTS
    'Slander':'TORTS',
    'Defamation':'TORTS',
    'Libel':'TORTS',
    'Trespass':'TORTS',
    'General Torts':'TORTS',
    'Negligence':'TORTS',
    'Conversion':'TORTS',
}


# ---------------------------
# LEVEL 1
# ---------------------------

L1_MAPPING = {
    k:v for k,v in L0_MAPPING.items()
    if v in ["CIVIL", "CRIMINAL"]
}


# ---------------------------
# LEVEL 2 CIVIL
# ---------------------------

L2_CIVIL_MAPPING = {

    'Appeal':'APPELLATE',
    'Writ':'APPELLATE',
    'Scire Facias':'APPELLATE',

    'Demurrer':'TRIAL_ENFORCEMENT',
    'Fieri Facias':'TRIAL_ENFORCEMENT',
    'General Execution':'TRIAL_ENFORCEMENT',
    'General Civil Procedure':'TRIAL_ENFORCEMENT',
    'Pleading':'TRIAL_ENFORCEMENT',
    'Capias':'TRIAL_ENFORCEMENT',
    'Attachment':'TRIAL_ENFORCEMENT',
}


# ---------------------------
# LEVEL 2 CRIMINAL
# ---------------------------

L2_CRIMINAL_MAPPING = {

    'General Criminal':'GENERAL_CRIMINAL',
    'Indictment':'GENERAL_CRIMINAL',

    'Homicide':'SPECIFIC_OFFENSE',
    'Assault & Battery':'SPECIFIC_OFFENSE',
    'Larceny':'SPECIFIC_OFFENSE',
    'Perjury':'SPECIFIC_OFFENSE',
    'Riot':'SPECIFIC_OFFENSE',
    'Counterfeiting':'SPECIFIC_OFFENSE',
    'Burglary/Robbery':'SPECIFIC_OFFENSE',
    'Affray':'SPECIFIC_OFFENSE',
}


# ============================================================
# APPLY LABELS TO REAL DATAFRAME
# ============================================================

df = df.copy()

df["L0_Label"] = df["Sub_Type"].map(L0_MAPPING).fillna("OTHER")

df["L1_Label"] = df["Sub_Type"].map(L1_MAPPING)

df["L2_Civil"] = df["Sub_Type"].map(L2_CIVIL_MAPPING)

df["L2_Criminal"] = df["Sub_Type"].map(L2_CRIMINAL_MAPPING)


# ============================================================
# REMOVE MISSING TEXT
# ============================================================

df = df.dropna(subset=["Case_Text"])

df["Case_Text"] = df["Case_Text"].astype(str)


print("DATA READY")
print(df.shape)

print("\nL0 DISTRIBUTION\n")
print(df["L0_Label"].value_counts())

DATA READY
(16091, 24)

L0 DISTRIBUTION

L0_Label
CIVIL       5542
CRIMINAL    4160
CONTRACT    2402
TORTS       2055
PROPERTY    1880
OTHER         52
Name: count, dtype: int64


In [20]:
# ╔══════════════════════════════════════════════════════════════╗
# ║ CELL 2 — MODEL PIPELINE + TRAINING FUNCTION                ║
# ╚══════════════════════════════════════════════════════════════╝

# ============================================================
# BUILD TEXT PIPELINE
# ============================================================

def build_pipeline():

    pipeline = Pipeline([

        (
            "tfidf",

            TfidfVectorizer(

                max_features=15000,

                ngram_range=(1,2),

                min_df=2,

                max_df=0.95,

                lowercase=True,

                strip_accents="unicode",

                sublinear_tf=True,

                stop_words="english"
            )
        ),

        (
            "classifier",

            LogisticRegression(

                max_iter=2000,

                class_weight="balanced",

                solver="lbfgs",

                multi_class="multinomial",

                random_state=RANDOM_STATE
            )
        )
    ])

    return pipeline


# ============================================================
# TRAINING FUNCTION
# ============================================================

def train_level(df_subset, label_col, level_name):

    print("\n" + "═"*70)
    print(level_name)
    print("═"*70)

    # remove null labels
    df_subset = df_subset.dropna(subset=[label_col]).copy()

    # text
    X = df_subset["Case_Text"]

    # labels
    y = df_subset[label_col]

    print("\nCLASS DISTRIBUTION\n")
    print(y.value_counts())

    # ========================================================
    # TRAIN TEST SPLIT
    # ========================================================

    X_train, X_test, y_train, y_test = train_test_split(

        X,
        y,

        test_size=TEST_SIZE,

        stratify=y,

        random_state=RANDOM_STATE
    )

    # ========================================================
    # BUILD MODEL
    # ========================================================

    pipeline = build_pipeline()

    # train
    pipeline.fit(X_train, y_train)

    # predict
    y_pred = pipeline.predict(X_test)

    # ========================================================
    # EVALUATION
    # ========================================================

    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    print(f"\nHOLDOUT MACRO-F1 : {macro_f1:.4f}")

    # ========================================================
    # CROSS VALIDATION
    # ========================================================

    cv = StratifiedKFold(

        n_splits=CV_FOLDS,

        shuffle=True,

        random_state=RANDOM_STATE
    )

    cv_scores = cross_val_score(

        pipeline,
        X,
        y,

        cv=cv,

        scoring="f1_macro",

        n_jobs=-1
    )

    print(
        f"CV MACRO-F1      : "
        f"{cv_scores.mean():.4f} ± {cv_scores.std():.4f}"
    )

    # ========================================================
    # CLASSIFICATION REPORT
    # ========================================================

    print("\nCLASSIFICATION REPORT\n")

    print(classification_report(y_test, y_pred))

    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    labels = sorted(y.unique())

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=labels
    )

    cm_df = pd.DataFrame(
        cm,
        index=labels,
        columns=labels
    )

    print("\nCONFUSION MATRIX\n")

    print(cm_df)

    return {

        "pipeline":pipeline,

        "macro_f1":macro_f1,

        "cv_mean":cv_scores.mean(),

        "cv_std":cv_scores.std()
    }

In [21]:
# ╔══════════════════════════════════════════════════════════════╗
# ║ CELL 3 — TRAIN HIERARCHICAL CLASSIFIERS                    ║
# ╚══════════════════════════════════════════════════════════════╝

# ============================================================
# LEVEL 0 — 5 CLASS
# ============================================================

df_l0 = df[df["L0_Label"] != "OTHER"].copy()

l0_results = train_level(

    df_l0,

    "L0_Label",

    "L0 — CASE TYPE CLASSIFIER"
)


# ============================================================
# LEVEL 1 — CIVIL vs CRIMINAL
# ============================================================

df_l1 = df[df["L1_Label"].notna()].copy()

l1_results = train_level(

    df_l1,

    "L1_Label",

    "L1 — CIVIL vs CRIMINAL"
)


# ============================================================
# LEVEL 2a — CIVIL SUBCLASSIFIER
# ============================================================

df_civil = df[df["L0_Label"] == "CIVIL"].copy()

l2_civil_results = train_level(

    df_civil,

    "L2_Civil",

    "L2a — APPELLATE vs TRIAL_ENFORCEMENT"
)


# ============================================================
# LEVEL 2b — CRIMINAL SUBCLASSIFIER
# ============================================================

df_criminal = df[df["L0_Label"] == "CRIMINAL"].copy()

l2_criminal_results = train_level(

    df_criminal,

    "L2_Criminal",

    "L2b — GENERAL_CRIMINAL vs SPECIFIC_OFFENSE"
)


# ============================================================
# SAVE MODELS
# ============================================================

l0_pipeline = l0_results["pipeline"]

l1_pipeline = l1_results["pipeline"]

l2_civil_pipeline = l2_civil_results["pipeline"]

l2_criminal_pipeline = l2_criminal_results["pipeline"]


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "═"*70)
print("FINAL RESULTS")
print("═"*70)

print(
    f"L0 F1  : {l0_results['macro_f1']:.4f}"
)

print(
    f"L1 F1  : {l1_results['macro_f1']:.4f}"
)

print(
    f"L2a F1 : {l2_civil_results['macro_f1']:.4f}"
)

print(
    f"L2b F1 : {l2_criminal_results['macro_f1']:.4f}"
)


══════════════════════════════════════════════════════════════════════
L0 — CASE TYPE CLASSIFIER
══════════════════════════════════════════════════════════════════════

CLASS DISTRIBUTION

L0_Label
CIVIL       5542
CRIMINAL    4160
CONTRACT    2402
TORTS       2055
PROPERTY    1880
Name: count, dtype: int64

HOLDOUT MACRO-F1 : 0.7237
CV MACRO-F1      : 0.7298 ± 0.0063

CLASSIFICATION REPORT

              precision    recall  f1-score   support

       CIVIL       0.81      0.75      0.78      1109
    CONTRACT       0.62      0.71      0.66       480
    CRIMINAL       0.84      0.69      0.76       832
    PROPERTY       0.69      0.76      0.72       376
       TORTS       0.62      0.81      0.70       411

    accuracy                           0.74      3208
   macro avg       0.72      0.74      0.72      3208
weighted avg       0.75      0.74      0.74      3208


CONFUSION MATRIX

          CIVIL  CONTRACT  CRIMINAL  PROPERTY  TORTS
CIVIL       829       134        44        

In [22]:
# ╔══════════════════════════════════════════════════════════════╗
# ║ CELL 4 — HIERARCHICAL PREDICTION                           ║
# ╚══════════════════════════════════════════════════════════════╝

# ============================================================
# HIERARCHICAL PREDICTION FUNCTION
# ============================================================

def hierarchical_predict(case_text):

    # ========================================================
    # LEVEL 0
    # ========================================================

    l0_probs = l0_pipeline.predict_proba([case_text])[0]

    l0_idx = np.argmax(l0_probs)

    l0_pred = l0_pipeline.classes_[l0_idx]

    l0_conf = l0_probs[l0_idx]

    result = {

        "L0_Predicted":l0_pred,
        "L0_Confidence":round(float(l0_conf),4),

        "L1_Predicted":None,
        "L1_Confidence":None,

        "L2_Predicted":None,
        "L2_Confidence":None
    }

    # ========================================================
    # CONFIDENCE THRESHOLD
    # ========================================================

    if l0_conf < CONFIDENCE_THRESHOLD:

        result["L0_Predicted"] = "OTHER"

        return result

    # ========================================================
    # NO HIERARCHY FOR THESE
    # ========================================================

    if l0_pred in ["CONTRACT", "PROPERTY", "TORTS"]:

        return result

    # ========================================================
    # LEVEL 1
    # ========================================================

    l1_probs = l1_pipeline.predict_proba([case_text])[0]

    l1_idx = np.argmax(l1_probs)

    l1_pred = l1_pipeline.classes_[l1_idx]

    l1_conf = l1_probs[l1_idx]

    result["L1_Predicted"] = l1_pred

    result["L1_Confidence"] = round(float(l1_conf),4)

    # ========================================================
    # LEVEL 2 ROUTING
    # ========================================================

    if l1_pred == "CIVIL":

        probs = l2_civil_pipeline.predict_proba([case_text])[0]

        idx = np.argmax(probs)

        pred = l2_civil_pipeline.classes_[idx]

        conf = probs[idx]

    else:

        probs = l2_criminal_pipeline.predict_proba([case_text])[0]

        idx = np.argmax(probs)

        pred = l2_criminal_pipeline.classes_[idx]

        conf = probs[idx]

    result["L2_Predicted"] = pred

    result["L2_Confidence"] = round(float(conf),4)

    return result


# ============================================================
# TEST ON RANDOM REAL CASES
# ============================================================

sample_df = df.sample(10, random_state=42)

rows = []

for _, row in sample_df.iterrows():

    pred = hierarchical_predict(row["Case_Text"])

    rows.append({

        "True_Sub_Type":row["Sub_Type"],

        "L0_Predicted":pred["L0_Predicted"],
        "L1_Predicted":pred["L1_Predicted"],
        "L2_Predicted":pred["L2_Predicted"],

        "L0_Confidence":pred["L0_Confidence"],
        "L1_Confidence":pred["L1_Confidence"],
        "L2_Confidence":pred["L2_Confidence"]
    })

pred_df = pd.DataFrame(rows)

print(pred_df)


# ============================================================
# PREDICT NEW UNSEEN TEXT
# ============================================================

new_text = """
the defendant unlawfully took property belonging
to another person with intent to permanently deprive
the lawful owner of possession
"""

prediction = hierarchical_predict(new_text)

print("\nNEW CASE PREDICTION\n")

for k,v in prediction.items():

    print(f"{k}: {v}")

             True_Sub_Type L0_Predicted L1_Predicted       L2_Predicted  \
0            General Torts        TORTS         None               None   
1                 Demurrer        CIVIL        CIVIL  TRIAL_ENFORCEMENT   
2         General Criminal     CRIMINAL     CRIMINAL   GENERAL_CRIMINAL   
3               Negligence        TORTS         None               None   
4                Ejectment     PROPERTY         None               None   
5         General Criminal        OTHER         None               None   
6  General Civil Procedure     CRIMINAL     CRIMINAL   GENERAL_CRIMINAL   
7         General Criminal     CRIMINAL     CRIMINAL   GENERAL_CRIMINAL   
8                     Bond     CONTRACT         None               None   
9         General Criminal     CRIMINAL     CRIMINAL   GENERAL_CRIMINAL   

   L0_Confidence  L1_Confidence  L2_Confidence  
0         0.7141            NaN            NaN  
1         0.4141         0.8935         0.9711  
2         0.6381         0.